# Submit từ toàn bộ dữ liệu đã xử lý — giữ URL cũ

**Chạy trên Colab, chọn T4 GPU → Run all.** Cell 2 mặc định chỉ khôi phục
checkpoint bằng manifest/báo cáo JSON. Không mở lại corpus, vector hoặc catalog/BM25.
Nếu báo cáo thiếu hoặc không khớp, cell 2 dừng; không tự chạy preparation lại.
Với dữ liệu mới chưa chuẩn bị, chủ động bật `ALLOW_CPU_PREPARATION` ở cell 2
và chạy CPU trước. Khi có `CPU_PREPARATION_COMPLETE`, tắt lại tùy chọn này,
đổi sang T4 GPU và Run all cùng notebook.
Đổi runtime mất biến/ổ tạm; các checkpoint đã publish trên Drive vẫn được giữ và tự nhận lại.
`/content/vmb_submit_work` là cache tạm trên máy chủ Colab. Dữ liệu gốc, checkpoint và ZIP
lưu bền trên Drive; notebook không tự tải ZIP hoặc dữ liệu về máy tính người dùng.
BGE đã có manifest COMPLETE thì bỏ lượt quét receipt/map trước inference.
Dùng lại đúng cache vector của VM; file thiếu tải qua API tối đa bốn file đồng thời,
kiểm SHA ngay lúc tải. Mapping và mẫu ANN có checkpoint gộp riêng trên Drive.
Khi cần sửa view chưa hoàn tất, bảng tra receipt được lưu nén qua API tại các mốc 256/512/1024/...,
khi inventory hoàn tất và khi đóng stage có dữ liệu mới. Chờ log `SAVED_ON_DRIVE`
để xác nhận đã lưu bền; `local checkpoint committed` chỉ xác nhận ổ tạm.
Tài khoản mới khôi phục bảng tra theo ID folder dùng chung rồi chỉ đọc receipt mới.
Khi đã có freeze receipt, resume bỏ qua vòng chọn/chép toàn bộ source part.
Spool local còn thì dùng lại; runtime mới chỉ khôi phục document/child cần để dựng spool,
chưa tải parent/section/ledger ở bước Index inputs. Log ghi riêng part reused/new.
Resume dùng cursor trên Drive để bỏ qua nội dung input part cũ. Runtime mới vẫn đọc source,
nhưng spool chỉ giữ hash sau cursor. SHA256 input cũ được kiểm khi bước tiếp theo dùng file.
Catalog tự dùng tối đa 4 CPU process, giữ nguyên PyVi/Jieba và BM25. Mỗi shard có
checkpoint tách từ trên Drive; SQLite tiếp tục từ shard đã commit nếu còn cùng runtime.
Nếu mất runtime khi catalog còn dở, phần SQL chưa có catalog hoàn tất cần phục hồi;
checkpoint tách từ vẫn dùng lại, không tính embedding lại.
Catalog đã hoàn tất của nguồn cũ được kế thừa khi receipt xác nhận toàn bộ nguồn vẫn còn:
giữ BM25/document/child/parent cũ, chỉ dựng lại ánh xạ dense position và index phần mới.
Log `Catalog reuse` và `Disk catalog/BM25 delta` ghi riêng phần cũ/mới; CPU vẫn cần cho delta.
Khôi phục catalog xác minh SHA ngay lúc giải nén. Bước kế thừa báo phase mỗi 15 giây,
lưu cursor chuyển ánh xạ theo khối 65.536 dòng; ngắt cùng runtime tiếp tục khối đã commit.
Không đọc/quick_check toàn catalog thêm lần nữa trong bước kế thừa. Catalog cuối vẫn được kiểm đầy đủ.
Chỉ cần chỉnh `DATA_ROOT_PATH` nếu Drive của tài khoản đang chạy dùng đường dẫn khác.

Giữ URL thuộc snapshot gốc, kể cả URL bị BTC loại trong bản dataset mới, theo yêu cầu
của Sếp. Không xóa dữ liệu gốc, không crawl/extract/chunk lại. Các query và ID gắn
đúng snapshot nguồn; lượt này **chưa xác nhận tương thích phạm vi BTC mới**.

Trên GPU: bổ sung BGE còn thiếu → lưu ZIP BGE/BM25 đủ 1.200 query trước →
dịch EN/ZH + PICO/subqueries/HyDE + document dense + Qwen rerank hai tầng +
source parents → ZIP nâng cấp. Qwen child dense dùng khi cache đầy đủ;
checkpoint Qwen chưa đủ vẫn được giữ. Đây chưa phải hoàn tất toàn bộ master plan.
Mỗi model ≤15B. Chưa có nhãn độc lập để fine-tune/hiệu chỉnh hoặc cam kết điểm.

Mỗi thời điểm chỉ **một phiên notebook này** ghi cùng DATA_ROOT. Các stage, vector,
model input, query và ZIP lưu bền trên Drive. Nếu bị ngắt, Run all cùng notebook
và cấu hình để tiếp tục checkpoint. Khi đã có ZIP baseline, có thể nộp thử trong
lúc lượt strong còn đang chạy; ZIP mạnh hơn chỉ xuất sau khi đủ 1.200 query.

In [ ]:
#@title 1. Drive, code và môi trường (tự nhận CPU/GPU)
"""Repair a nonempty, unmounted Colab Drive directory without deleting files.

Pure standard library so notebook setup can embed this before pip/package imports.
Kernel mount state must positively exclude this directory and any nested mounts
before an atomic rename preserves the complete local directory under /content.
"""
import os
from pathlib import Path
import re
from uuid import uuid4


def preserve_unmounted_drive_directory(mount, *, content_root='/content',
                                       mountinfo_file='/proc/self/mountinfo'):
    target, parent = Path(mount).absolute(), Path(content_root).resolve()
    if target.name != 'drive' or target.parent.resolve() != parent:
        raise ValueError('Drive recovery may preserve only the named content_root/drive directory.')
    if target.is_symlink():
        raise RuntimeError('Drive mountpoint is a symlink; refusing to move it.')
    try:
        entries = Path(mountinfo_file).read_text(encoding='utf-8').splitlines()
    except OSError as error:
        raise RuntimeError('Cannot verify kernel mount state; no Drive directory was moved.') from error
    if not entries:
        raise RuntimeError('Empty kernel mount information; no Drive directory was moved.')
    for line in entries:
        fields = line.split()
        if len(fields) < 10 or '-' not in fields[6:]:
            raise RuntimeError('Invalid kernel mount information; no Drive directory was moved.')
        name = re.sub(r'\\([0-7]{3})', lambda match: chr(int(match[1], 8)), fields[4])
        point = Path(name).absolute()
        if point == target or point.is_relative_to(target):
            raise RuntimeError(f'Kernel mount still active at {point}; refusing to move Drive data.')
    if os.path.ismount(target):
        raise RuntimeError('Drive is still a mountpoint; refusing to move it.')
    if not target.exists():
        return None
    if not target.is_dir():
        raise RuntimeError('Drive mountpoint is a file; refusing to move it.')
    if next(target.iterdir(), None) is None:
        return None
    backup = parent / ('vmb_drive_unmounted_' + uuid4().hex)
    if backup.exists() or backup.is_symlink():
        raise RuntimeError('Preservation path already exists; no Drive directory was moved.')
    # Same-parent rename retains every nested file and never follows child links.
    # Only an unmounted local directory is moved; original remote data is untouched.
    target.rename(backup)
    target.mkdir()
    print('Local unmounted Drive directory preserved at:', backup, flush=True)
    return backup


def safe_colab_mount(mount='/content/drive', *, force_remount=False, readonly=False,
                     drive_api=None, content_root='/content', mountinfo_file='/proc/self/mountinfo'):
    if drive_api is None:
        from google.colab import drive as drive_api
    try:
        drive_api.mount(str(mount), force_remount=force_remount, readonly=readonly)
        return None
    except ValueError as error:
        if 'Mountpoint must not already contain files' not in str(error):
            raise
    backup = preserve_unmounted_drive_directory(mount, content_root=content_root,
        mountinfo_file=mountinfo_file)
    drive_api.mount(str(mount), force_remount=True, readonly=readonly)
    return backup


def safe_colab_remount(mount, *, readonly=False, drive_api=None,
                       content_root='/content', mountinfo_file='/proc/self/mountinfo'):
    if drive_api is None:
        from google.colab import drive as drive_api
    try:
        drive_api.flush_and_unmount()
    except Exception as error:
        print('Drive flush/unmount did not complete:', type(error).__name__, flush=True)
    return safe_colab_mount(mount, force_remount=True, readonly=readonly, drive_api=drive_api,
        content_root=content_root, mountinfo_file=mountinfo_file)


import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys

DATA_ROOT_PATH = "/content/drive/MyDrive/VietMedBridge/data" #@param {type:"string"}
DATA_ROOT = Path(DATA_ROOT_PATH)
WORK_DIR = Path("/content/vmb_submit_work")
CHECKOUT = Path("/content/VietMedBridge-submit")
REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
WORKFLOW_VERSION = "all-available-legacy-20261017-v9"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK_DIR.mkdir(parents=True,exist_ok=True)

if not Path("/var/colab/hostname").exists():
    raise RuntimeError("Notebook này chạy trên Google Colab. Kaggle dùng notebook embedding worker riêng.")
from google.colab import drive, userdata
safe_colab_mount("/content/drive")
# The shared-folder shortcut may exist only under its target ID on another account.
bootstrap_roots = list(dict.fromkeys([DATA_ROOT,
    Path("/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge/data"),
    Path("/content/drive/MyDrive/VietMedBridge/data")]))
for bootstrap_root in bootstrap_roots:
    try:
        json.loads((bootstrap_root / "active_data_candidate.json").read_text())
    except FileNotFoundError:
        continue
    except OSError as error:
        if error.errno in (103,107):
            break  # The preflight below reconnects before code/model setup.
        raise
    DATA_ROOT = bootstrap_root
    break
try:
    (DATA_ROOT / "active_data_candidate.json").read_bytes()
except OSError as error:
    if error.errno not in (103,107):
        raise
    print("Drive preflight: remount trước khi đọc code lock hoặc nạp model.",flush=True)
    safe_colab_remount(Path("/content/drive"))
    (DATA_ROOT / "active_data_candidate.json").read_bytes()
if not (DATA_ROOT / "active_data_candidate.json").is_file():
    raise RuntimeError("DATA_ROOT chưa trỏ tới data đã freeze. Chỉnh DATA_ROOT ở đầu cell, không tạo data rỗng.")
print("DATA_ROOT:", DATA_ROOT,flush=True)
# A lock for this workflow, separate from old notebook/data-processing locks.
code_lock = DATA_ROOT / "retrieval" / (WORKFLOW_VERSION+"-code-lock.json")
locked = json.loads(code_lock.read_text()) if code_lock.exists() else None
if locked and (locked.get("repo") != REPO_URL or not re.fullmatch(r"[0-9a-f]{40}",locked.get("commit",""))):
    raise ValueError("Submission code lock không hợp lệ.")
reference = locked["commit"] if locked else "origin/main"
def command(args):
    print("Setup:"," ".join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),check=True)
if not (CHECKOUT / ".git").is_dir():
    command(["git","clone",REPO_URL,CHECKOUT])
command(["git","-C",CHECKOUT,"fetch","origin"])
command(["git","-C",CHECKOUT,"checkout","--detach",reference])
COMMIT = subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
if not (CHECKOUT / "src/vietmedbridge/submission_workflow.py").is_file():
    raise RuntimeError("Checkout chưa có workflow submission mới. Dùng notebook cùng bản code đã push.")
# Detect the physical GPU before importing torch/transformers.
GPU_SESSION = False
if shutil.which("nvidia-smi"):
    probe = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True)
    GPU_SESSION = probe.returncode == 0 and bool(probe.stdout.strip())
    print("Physical GPUs:",probe.stdout.strip() or "none",flush=True)
runtime = None
runtime_lock = DATA_ROOT / "retrieval_embedding_runtime_lock.json"
if GPU_SESSION:
    if not runtime_lock.is_file():
        raise RuntimeError("Thiếu runtime lock của vector đã nhập. Khôi phục retrieval_embedding_runtime_lock.json trước khi inference.")
    lock = json.loads(runtime_lock.read_text())
    payload = {k:v for k,v in lock.items() if k != "manifest_sha256"}
    checksum = hashlib.sha256(json.dumps(payload,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()
    if checksum != lock.get("manifest_sha256"):
        raise ValueError("Embedding runtime lock đã thay đổi.")
    runtime = lock["runtime"]
    if (set(runtime) != {"torch","transformers","bitsandbytes"}
        or any(not re.fullmatch(r"[0-9][A-Za-z0-9.+_-]*",v) for v in runtime.values())
        or not re.search(r"\+cu[0-9]+$",runtime["torch"])):
        raise ValueError("Runtime lock cần CUDA torch, transformers và bitsandbytes hợp lệ.")
    for name,expected in lock["model_code"].items():
        if Path(name).name != name or hashlib.sha256((CHECKOUT / "src/vietmedbridge" / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Model producer code khác cache đã tính; dừng trước khi nạp model.")
    if "torch" in sys.modules and sys.modules["torch"].__version__ != runtime["torch"]:
        raise RuntimeError("Runtime đang giữ torch phiên bản cũ. Restart session rồi Run all; giữ nguyên Drive/checkpoint.")
    try:
        torch_version = importlib.metadata.version("torch")
    except importlib.metadata.PackageNotFoundError:
        torch_version = None
    if torch_version != runtime["torch"]:
        suffix = re.search(r"\+(cu[0-9]+)$",runtime["torch"])[1]
        command([sys.executable,"-m","pip","install","torch=="+runtime["torch"],"--index-url","https://download.pytorch.org/whl/"+suffix])
        command([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"])
    constraints = WORK_DIR / "runtime-constraints.txt"
    constraints.write_text("".join(f"{name}=={version}\n" for name,version in runtime.items()))
    command([sys.executable,"-m","pip","install","-c",constraints,"-e",str(CHECKOUT)+"[retrieval,strong]"])
else:
    command([sys.executable,"-m","pip","install","-e",CHECKOUT,
        "numpy>=1.26,<3","faiss-cpu==1.12.0","pyvi==0.1.1","jieba==0.42.1",
        "scikit-learn==1.7.2","sklearn-crfsuite==0.5.0"])
sys.path.insert(0,str(CHECKOUT / "src"))
if importlib.util.find_spec("googleapiclient") is None:
    command([sys.executable,"-m","pip","install","google-api-python-client>=2,<3"])
import vietmedbridge
if Path(vietmedbridge.__file__).resolve() != (CHECKOUT / "src/vietmedbridge/__init__.py").resolve():
    raise RuntimeError("Kernel đang giữ package của checkout khác. Restart session rồi Run all.")
importlib.invalidate_caches()
for module_name in ("vietmedbridge.artifacts","vietmedbridge.colab_storage","vietmedbridge.content_embeddings",
                    "vietmedbridge.input_resume","vietmedbridge.input_spool","vietmedbridge.index_inputs",
                    "vietmedbridge.catalog_reuse","vietmedbridge.catalog_builder","vietmedbridge.catalog_storage",
                    "vietmedbridge.official_scope","vietmedbridge.submission_workflow"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
from vietmedbridge.artifacts import atomic_json, read_json, sha256_file, verify_file
if not locked:
    atomic_json(code_lock,{"repo":REPO_URL,"commit":COMMIT,"workflow":WORKFLOW_VERSION})
try:
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass # Public models do not require a token.
if GPU_SESSION:
    import torch
    if not torch.cuda.is_available() or torch.__version__ != runtime["torch"]:
        raise RuntimeError("CUDA torch chưa hoạt động. Restart session rồi Run all sau khi setup đã cài xong.")
    print("CUDA:",torch.__version__,torch.cuda.get_device_name(0),flush=True)
print("Code:",COMMIT,"| package:",vietmedbridge.__file__,flush=True)
print("Phase:","GPU inference" if GPU_SESSION else "CPU corpus/catalog preparation",flush=True)


In [ ]:
#@title 2. Chỉ khôi phục checkpoint JSON; không tự chạy preparation lại
ALLOW_CPU_PREPARATION = False #@param {type:"boolean"}
"""Restore a sealed submission handoff using JSON only; never start CPU preparation.

Embedded in the notebook so old producer/code locks and corpus fingerprints stay
unchanged. This checks identities, not catalog/vector payload integrity. Native
readers still validate payloads when inference actually consumes them.
"""
from pathlib import Path

from vietmedbridge.artifacts import atomic_json, digest_json, read_json
from vietmedbridge.official_scope import discover_sources
from vietmedbridge.scale_benchmark import load_handoff


def restore_submission_preparation_only(data_root, checkout):
    root, checkout = Path(data_root), Path(checkout)
    print("RESUME_ONLY: đọc manifest/báo cáo JSON; không mở corpus, vector hoặc catalog/BM25.", flush=True)
    build, candidate, inputs = load_handoff(root)
    config_sha = digest_json(read_json(checkout / "configs/retrieval_full.json"))
    source_names = discover_sources(root)
    corpus_sha = read_json(root / "raw/snapshot.json")["files"]["links_corpus.parquet"]["sha256"]
    identity = {"snapshot_sha256": candidate["snapshot_sha256"],
        "candidate_manifest_sha256": candidate["candidate_manifest_sha256"],
        "index_inputs_manifest_sha256": inputs["manifest_sha256"]}
    pointer = root / "retrieval/current_submission.json"
    reports = [pointer, build / "submission_preparation.json",
        root / "retrieval/cpu_preparation" / (candidate["candidate_manifest_sha256"][:16] + ".json")]
    for path in reports:
        if not path.is_file():
            continue
        saved = read_json(path)
        if (saved.get("state") != "CPU_PREPARATION_COMPLETE"
            or saved.get("build_run") != build.name
            or saved.get("candidate_manifest_sha256") != candidate["candidate_manifest_sha256"]
            or saved.get("retrieval_config_sha256") != config_sha
            or saved.get("official_sha256") != candidate["origin_corpus_sha256"]
            or candidate.get("input_kind") != "AVAILABLE_CORPUS"
            or saved.get("legacy_urls_retained") is not True
            or saved.get("source_builds") != source_names
            or any(saved.get("catalog", {}).get(k) != v for k, v in identity.items())
            or corpus_sha != candidate["origin_corpus_sha256"]):
            continue
        result = saved | {"results_root": str(root.parent / "submission_runs" / build.name)}
        if not pointer.is_file() or read_json(pointer) != result:
            atomic_json(pointer, result)  # only a small handoff pointer; no stage output is rebuilt
        print(f"RESUME_READY: {build.name} | {saved['documents']:,} documents | CPU preparation reused.", flush=True)
        return result
    raise RuntimeError(
        "RESUME_ONLY_STOP: thiếu hoặc không khớp báo cáo CPU_PREPARATION_COMPLETE. "
        "Chưa đọc lại corpus/vector/catalog và chưa bắt đầu preparation. "
        "Kiểm tra DATA_ROOT/candidate/code lock; chỉ bật ALLOW_CPU_PREPARATION khi chủ động chuẩn bị dữ liệu mới.")


if ALLOW_CPU_PREPARATION:
    from vietmedbridge.submission_workflow import ensure_submission_preparation
    PREPARATION = ensure_submission_preparation(DATA_ROOT,CHECKOUT,WORK_DIR,gpu_session=GPU_SESSION)
else:
    PREPARATION = restore_submission_preparation_only(DATA_ROOT,CHECKOUT)
print(json.dumps({k: PREPARATION[k] for k in (
    "state", "build_run", "candidate_manifest_sha256", "documents", "unique_dense_inputs"
)},ensure_ascii=False,indent=2),flush=True)
if GPU_SESSION:
    print("Tiếp tục cell 3. Cell 2 chưa kiểm payload; cell inference kiểm checkpoint khi sử dụng.",flush=True)
else:
    print("CPU handoff đã có. Chọn GPU rồi chạy cell 1 → 2 → 3; giữ ALLOW_CPU_PREPARATION=False khi resume.",flush=True)


In [ ]:
#@title 3. BGE phần còn thiếu và ZIP đầu tiên đủ 1.200 query (GPU)
"""Repair a nonempty, unmounted Colab Drive directory without deleting files.

Pure standard library so notebook setup can embed this before pip/package imports.
Kernel mount state must positively exclude this directory and any nested mounts
before an atomic rename preserves the complete local directory under /content.
"""
import os
from pathlib import Path
import re
from uuid import uuid4


def preserve_unmounted_drive_directory(mount, *, content_root='/content',
                                       mountinfo_file='/proc/self/mountinfo'):
    target, parent = Path(mount).absolute(), Path(content_root).resolve()
    if target.name != 'drive' or target.parent.resolve() != parent:
        raise ValueError('Drive recovery may preserve only the named content_root/drive directory.')
    if target.is_symlink():
        raise RuntimeError('Drive mountpoint is a symlink; refusing to move it.')
    try:
        entries = Path(mountinfo_file).read_text(encoding='utf-8').splitlines()
    except OSError as error:
        raise RuntimeError('Cannot verify kernel mount state; no Drive directory was moved.') from error
    if not entries:
        raise RuntimeError('Empty kernel mount information; no Drive directory was moved.')
    for line in entries:
        fields = line.split()
        if len(fields) < 10 or '-' not in fields[6:]:
            raise RuntimeError('Invalid kernel mount information; no Drive directory was moved.')
        name = re.sub(r'\\([0-7]{3})', lambda match: chr(int(match[1], 8)), fields[4])
        point = Path(name).absolute()
        if point == target or point.is_relative_to(target):
            raise RuntimeError(f'Kernel mount still active at {point}; refusing to move Drive data.')
    if os.path.ismount(target):
        raise RuntimeError('Drive is still a mountpoint; refusing to move it.')
    if not target.exists():
        return None
    if not target.is_dir():
        raise RuntimeError('Drive mountpoint is a file; refusing to move it.')
    if next(target.iterdir(), None) is None:
        return None
    backup = parent / ('vmb_drive_unmounted_' + uuid4().hex)
    if backup.exists() or backup.is_symlink():
        raise RuntimeError('Preservation path already exists; no Drive directory was moved.')
    # Same-parent rename retains every nested file and never follows child links.
    # Only an unmounted local directory is moved; original remote data is untouched.
    target.rename(backup)
    target.mkdir()
    print('Local unmounted Drive directory preserved at:', backup, flush=True)
    return backup


def safe_colab_mount(mount='/content/drive', *, force_remount=False, readonly=False,
                     drive_api=None, content_root='/content', mountinfo_file='/proc/self/mountinfo'):
    if drive_api is None:
        from google.colab import drive as drive_api
    try:
        drive_api.mount(str(mount), force_remount=force_remount, readonly=readonly)
        return None
    except ValueError as error:
        if 'Mountpoint must not already contain files' not in str(error):
            raise
    backup = preserve_unmounted_drive_directory(mount, content_root=content_root,
        mountinfo_file=mountinfo_file)
    drive_api.mount(str(mount), force_remount=True, readonly=readonly)
    return backup


def safe_colab_remount(mount, *, readonly=False, drive_api=None,
                       content_root='/content', mountinfo_file='/proc/self/mountinfo'):
    if drive_api is None:
        from google.colab import drive as drive_api
    try:
        drive_api.flush_and_unmount()
    except Exception as error:
        print('Drive flush/unmount did not complete:', type(error).__name__, flush=True)
    return safe_colab_mount(mount, force_remount=True, readonly=readonly, drive_api=drive_api,
        content_root=content_root, mountinfo_file=mountinfo_file)

"""API receipt reads and narrowly scoped lookup checkpoints, without Drive FUSE."""
import copy
import hashlib
import json
from pathlib import Path, PurePosixPath
import re
import threading
import time


class LookupPublishUnavailable(RuntimeError):
    """An optional lookup upload failed; paid vector receipts remain untouched."""


class RemoteReceipt:
    def __init__(self, source, metadata):
        self.source, self.metadata = source, metadata
        self.name = metadata['name']

    def read_json(self):
        return self.source.read_json(self.metadata)


class DriveReceiptSource:
    endpoint = 'https://www.googleapis.com/drive/v3/files'

    def __init__(self, session_factory, data_root, canonical_root, *,
                 data_folder_id='1Ez1S9jk6FO_puxC8r2vPSc0LJbuOkkdj', report=print):
        if not re.fullmatch(r'[A-Za-z0-9_-]+', data_folder_id):
            raise ValueError('Invalid Drive data folder ID.')
        self.session_factory, self.sessions = session_factory, threading.local()
        self.requested_root, self.data_root = Path(data_root).absolute(), Path(canonical_root)
        self.folder_id, self.report, self.folders = data_folder_id, report, {}
        self.payload_directories = {}

    def directory_files(self, relative_directory, *, refresh=False):
        """One paginated metadata listing, rather than one lookup per payload."""
        relative = PurePosixPath(relative_directory)
        if relative.is_absolute() or '..' in relative.parts or '\\' in relative_directory or not relative.parts:
            raise ValueError('Invalid payload directory.')
        if not refresh and relative_directory in self.payload_directories:
            return self.payload_directories[relative_directory]
        parent = self.folder_id
        for part in relative.parts:
            if (parent, part) not in self.folders:
                value = self.child(parent, part, folder=True)
                if value is None:
                    return {}
                self.folders[parent, part] = value['id']
            parent = self.folders[parent, part]
        result = {}
        for value in self._list(f"'{parent}' in parents and trashed=false"):
            if value['name'] in result:
                raise ValueError('Duplicate payload names in Drive directory.')
            result[value['name']] = value
        self.payload_directories[relative_directory] = result
        return result

    def payload_metadata(self, relative_path):
        relative = PurePosixPath(relative_path)
        if relative.is_absolute() or '..' in relative.parts or '\\' in relative_path or len(relative.parts) < 2:
            raise ValueError('Invalid immutable payload path.')
        directory = relative.parent.as_posix()
        result = self.directory_files(directory).get(relative.name)
        if result is None:
            result = self.directory_files(directory, refresh=True).get(relative.name)
        if result is None or 'size' not in result:
            raise ValueError('Expected immutable payload is missing from Drive.')
        return result

    def _session(self):
        if not hasattr(self.sessions, 'session'):
            self.sessions.session = self.session_factory()
        return self.sessions.session

    def _get(self, url, params, *, stream=False):
        import requests
        session = self._session()
        for attempt in range(4):
            try:
                response = session.get(url, params=params, stream=stream, timeout=(10,60))
            except requests.RequestException:
                if attempt == 3:
                    raise
            else:
                if response.status_code not in (429,500,502,503,504) or attempt == 3:
                    response.raise_for_status()
                    return response
                response.close()
            self.report(f'Drive receipt API: transient request failure; retry {attempt+1}/3')
            time.sleep(2**attempt)

    def _list(self, query):
        token, seen = None, set()
        while True:
            params = {'q': query, 'pageSize': 1000, 'supportsAllDrives': 'true',
                'includeItemsFromAllDrives': 'true',
                'fields': 'nextPageToken,incompleteSearch,files(id,name,mimeType,size,md5Checksum)'}
            if token:
                params['pageToken'] = token
            with self._get(self.endpoint, params) as response:
                page = response.json()
            if page.get('incompleteSearch'):
                raise ValueError('Drive receipt listing is incomplete; refusing partial inventory.')
            yield from page.get('files', [])
            token = page.get('nextPageToken')
            if not token:
                return
            if token in seen:
                raise ValueError('Drive receipt pagination repeated; inventory is incomplete.')
            seen.add(token)

    def child(self, parent, name, *, folder=False):
        safe = name.replace('\\', '\\\\').replace("'", "\\'")
        query = f"'{parent}' in parents and trashed=false and name='{safe}'"
        if folder:
            query += " and mimeType='application/vnd.google-apps.folder'"
        values = list(self._list(query))
        if len(values) > 1:
            raise ValueError('Duplicate Drive path names; refusing ambiguous receipt source.')
        return values[0] if values else None

    def relative(self, name):
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or '\\' in name or not relative.parts:
            raise ValueError('Invalid relative Drive receipt path.')
        parent = self.folder_id
        for part in relative.parts[:-1]:
            key = parent, part
            if key not in self.folders:
                value = self.child(parent, part, folder=True)
                if value is None:
                    return None
                self.folders[key] = value['id']
            parent = self.folders[key]
        return self.child(parent, relative.name)

    def download(self, metadata, target, *, expected_sha256=None, max_bytes=None):
        size = int(metadata['size'])
        if max_bytes is not None and size > max_bytes:
            raise ValueError('Receipt exceeds the bounded JSON size.')
        sha, md5, count = hashlib.sha256(), hashlib.md5(), 0
        with self._get(self.endpoint+'/'+metadata['id'], {'alt':'media','supportsAllDrives':'true'}, stream=True) as response:
            with Path(target).open('wb') as output:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    count += len(chunk)
                    if count > size or (max_bytes is not None and count > max_bytes):
                        raise ValueError('Drive receipt payload exceeds its declared size.')
                    sha.update(chunk)
                    md5.update(chunk)
                    output.write(chunk)
        if (count != size or (metadata.get('md5Checksum') and md5.hexdigest() != metadata['md5Checksum'])
            or (expected_sha256 and sha.hexdigest() != expected_sha256)):
            raise ValueError('Drive receipt size/checksum mismatch.')

    def read_json(self, metadata):
        import tempfile
        with tempfile.TemporaryDirectory(prefix='vmb-api-json-') as directory:
            path = Path(directory) / 'receipt.json'
            self.download(metadata, path, max_bytes=16*1024*1024)
            return json.loads(path.read_text(encoding='utf-8'))

    def relative_json(self, path):
        metadata = self.relative(path)
        return None if metadata is None else self.read_json(metadata)

    def verify_candidate(self, expected):
        active = self.relative_json('active_data_candidate.json')
        if active is None or active.get('candidate_manifest_sha256') != expected:
            raise ValueError('Drive API data folder differs from the expected frozen candidate.')
        candidate = self.relative_json('processed/'+active['build_run']+'/'+active['candidate_name'])
        from vietmedbridge.artifacts import digest_json
        if (candidate is None or candidate.get('candidate_manifest_sha256') != expected
            or digest_json({k:v for k,v in candidate.items() if k != 'candidate_manifest_sha256'}) != expected):
            raise ValueError('Drive API candidate digest changed.')
        self.active, self.candidate = active, candidate

    def receipts(self, identity):
        from vietmedbridge.artifacts import digest_json
        # Resolve the folder itself without relying on any receipt being present.
        cache = self.child(self.folder_id, 'model_cache', folder=True)
        folder = self.child(cache['id'], digest_json(identity)[:24], folder=True) if cache else None
        if folder is None:
            return []
        result, names, payloads = [], set(), {}
        # Drive name-contains is prefix matching, not a suffix predicate. List
        # the complete producer folder then filter receipt names locally.
        query = f"'{folder['id']}' in parents and trashed=false"
        for value in self._list(query):
            if value['name'] in payloads:
                raise ValueError('Duplicate names in content cache directory.')
            payloads[value['name']] = value
            if not value['name'].endswith('.done.json'):
                continue
            if (not re.fullmatch(r'block-[0-9a-f]{64}\.done\.json', value['name'])
                or value['name'] in names or 'size' not in value):
                raise ValueError('Invalid or duplicate API receipt filename.')
            names.add(value['name'])
            result.append(RemoteReceipt(self, value))
        self.payload_directories['model_cache/'+digest_json(identity)[:24]] = payloads
        self.report(f'Drive receipt API: complete listing, {len(result):,} receipts; no mount reads')
        return sorted(result, key=lambda value: value.name)

    def publish_cache_checkpoint(self, relative_marker, payload, marker):
        """Only mapping bundles and exact ANN training samples may be published here."""
        from vietmedbridge.artifacts import digest_json
        binding = marker.get('binding', {})
        source = binding.get('source_manifest_sha256', '')
        kind = binding.get('kind')
        if kind == 'mapping-bundle-v1':
            directory, prefix, suffix = 'retrieval/mapping_bundles/'+source, 'maps-', '.tar.gz'
        elif kind == 'index-sample-v1' and re.fullmatch(r'[0-9a-f]{64}', binding.get('positions_sha256', '')):
            directory = 'retrieval/index_samples/'+source+'/'+binding['positions_sha256']
            prefix, suffix = 'sample-', '.npy'
        else:
            raise ValueError('Unsupported cache checkpoint kind.')
        if (binding.get('data_folder_id') != self.folder_id or binding.get('schema_version') != 1
            or not re.fullmatch(r'[0-9a-f]{64}', source)
            or relative_marker != directory+'/current.json'
            or not re.fullmatch(r'[0-9a-f]{64}', marker.get('sha256', ''))
            or marker.get('path') != prefix+marker['sha256']+suffix
            or marker.get('manifest_sha256') != digest_json({k:v for k,v in marker.items() if k != 'manifest_sha256'})):
            raise ValueError('Cache checkpoint escapes its declared source/folder binding.')
        sha, md5 = hashlib.sha256(), hashlib.md5()
        with Path(payload).open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024*1024), b''):
                sha.update(chunk); md5.update(chunk)
        if sha.hexdigest() != marker['sha256']:
            raise ValueError('Cache checkpoint payload changed before upload.')
        try:
            return self._publish_lookup_files(directory, Path(payload), marker, md5.hexdigest())
        except ValueError:
            raise
        except Exception as error:
            raise LookupPublishUnavailable(type(error).__name__) from error

    def publish_lookup(self, relative_marker, payload, marker):
        """Upload a checksum-addressed lookup, then its pointer. Never write vector/source files."""
        from vietmedbridge.artifacts import digest_json
        body = {k:v for k,v in marker.items() if k != 'manifest_sha256'}
        binding = marker.get('binding', {})
        expected_directory = 'retrieval/content_lookups/' + digest_json(binding)[:16]
        if (relative_marker != expected_directory + '/current.json'
            or binding.get('schema_version') != 2 or binding.get('data_folder_id') != self.folder_id
            or marker.get('manifest_sha256') != digest_json(body)
            or marker.get('path') != 'lookup-' + marker.get('sha256', '') + '.sqlite.gz'
            or not re.fullmatch(r'lookup-[0-9a-f]{64}\.sqlite\.gz', marker.get('path', ''))):
            raise ValueError('Lookup upload is outside its bound checkpoint directory.')
        sha, md5 = hashlib.sha256(), hashlib.md5()
        with Path(payload).open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024*1024), b''):
                sha.update(chunk)
                md5.update(chunk)
        if sha.hexdigest() != marker['sha256']:
            raise ValueError('Lookup payload changed before upload.')
        try:
            return self._publish_lookup_files(expected_directory, Path(payload), marker, md5.hexdigest())
        except ValueError:
            raise
        except Exception as error:
            # Do not include provider messages/URLs: resumable upload URLs are private.
            raise LookupPublishUnavailable(type(error).__name__) from error

    def _publish_lookup_files(self, directory, payload, marker, md5):
        from vietmedbridge.artifacts import digest_json
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload
        import io
        service = build('drive', 'v3', credentials=copy.copy(self._session().credentials), cache_discovery=False)
        files = service.files()
        fields = 'id,name,mimeType,size,md5Checksum'

        def new_id():
            return files.generateIds(count=1, space='drive', type='files').execute(num_retries=3)['ids'][0]

        def ensure_folder(parent, name):
            value = self.child(parent, name, folder=True)
            if value is None:
                identifier = new_id()
                try:
                    value = files.create(body={'id':identifier, 'name':name, 'parents':[parent],
                        'mimeType':'application/vnd.google-apps.folder'}, fields=fields,
                        supportsAllDrives=True).execute(num_retries=3)
                except Exception:
                    # A lost response may follow a successful create. Reconcile the
                    # preallocated ID; never retry a folder with a different ID.
                    value = files.get(fileId=identifier, fields=fields, supportsAllDrives=True).execute(num_retries=3)
                    if value.get('name') != name or value.get('mimeType') != 'application/vnd.google-apps.folder':
                        raise ValueError('Created lookup folder identity changed.')
            self.folders[parent, name] = value['id']
            return value['id']

        parent = self.folder_id
        for name in PurePosixPath(directory).parts:
            parent = ensure_folder(parent, name)
        current = self.child(parent, 'current.json')
        if current is not None:
            existing = self.read_json(current)
            if (existing.get('manifest_sha256') != digest_json(
                    {k:v for k,v in existing.items() if k != 'manifest_sha256'})
                or existing.get('binding') != marker['binding']):
                raise ValueError('Existing lookup checkpoint binding/digest changed.')
            count_key = 'receipts' if 'receipts' in marker else 'parts'
            if existing[count_key] > marker[count_key]:
                return existing  # An older local prefix must not replace a newer saved lookup.

        def upload(media, name, existing=None):
            if existing is None:
                request = files.create(body={'id':new_id(), 'name':name, 'parents':[parent]},
                    media_body=media, fields=fields, supportsAllDrives=True)
            else:
                request = files.update(fileId=existing['id'], media_body=media,
                    fields=fields, supportsAllDrives=True)
            result = None
            while result is None:
                _, result = request.next_chunk(num_retries=3)
            # Query provider metadata independently before publishing a pointer.
            return files.get(fileId=result['id'], fields=fields, supportsAllDrives=True).execute(num_retries=3)

        saved = self.child(parent, marker['path'])
        if saved is None:
            mime = 'application/gzip' if marker['path'].endswith('.gz') else 'application/octet-stream'
            saved = upload(MediaFileUpload(str(payload), mimetype=mime,
                chunksize=8*1024*1024, resumable=True), marker['path'])
        if int(saved.get('size', -1)) != payload.stat().st_size or saved.get('md5Checksum') != md5:
            raise ValueError('Uploaded lookup payload size/checksum mismatch; pointer not advanced.')
        data = json.dumps(marker, ensure_ascii=False, sort_keys=True).encode('utf-8')
        saved_marker = upload(MediaIoBaseUpload(io.BytesIO(data), mimetype='application/json',
            chunksize=256*1024, resumable=True), 'current.json', current)
        if int(saved_marker.get('size', -1)) != len(data) or saved_marker.get('md5Checksum') != hashlib.md5(data).hexdigest():
            raise ValueError('Uploaded lookup pointer size/checksum mismatch.')
        return marker


def authenticate_drive_receipts(data_root, expected_candidate_sha256):
    """Authenticate on the main Colab thread; never print/save credential tokens."""
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    root = Path(data_root).resolve()
    print('Drive receipts: authenticate API; receipts read-only, lookup checkpoints saved in retrieval/content_lookups', flush=True)
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    client = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data_root, root)
    client.verify_candidate(expected_candidate_sha256)
    print('Drive receipts: API source verified against the same frozen candidate', flush=True)
    return client

"""Cache-aware submission I/O; immutable producer/model source files stay pinned."""
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
import hashlib
import os
from pathlib import Path, PurePosixPath
import shutil
import sqlite3
import tarfile
import threading
import time
import uuid

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from vietmedbridge.artifacts import atomic_json, digest_json, read_json, sha256_file, verify_file
from vietmedbridge.content_embeddings import ContentVectorParts as OriginalVectorParts, checked, seal
from vietmedbridge.embeddings import validate_vectors
from vietmedbridge.portable_embeddings import resources_relative

if 'LookupPublishUnavailable' not in globals():
    from scripts.drive_receipts_api import LookupPublishUnavailable


def read_declared_vector_manifest(data_root, candidate, inputs, config, *, family='bge', api=None, progress=print):
    """Use the sealed manifest, validating actual payloads only when consumed.

    None means the caller must discover partial BGE receipts through the native
    fallback. A declared incomplete Qwen view can only defer that optional leg.
    """
    if family not in ('bge', 'qwen'):
        raise ValueError('Unknown embedding family.')
    checked(inputs)
    view = Path(data_root) / resources_relative(candidate, config) / family
    def load(name):
        if api is not None:
            return api.relative_json((view / name).relative_to(Path(data_root)).as_posix())
        path = view / name
        return read_json(path) if path.is_file() else None
    identity = load('config.json')
    if identity is None:
        return None
    spec = config['dense' if family == 'bge' else 'second_dense']
    if (identity.get('input_manifest_sha256') != inputs['manifest_sha256']
        or identity.get('input_count') != inputs['input_count']
        or identity.get('format') != 'content-reference-v1'
        or identity.get('dimension') != (1024 if family == 'bge' else 4096)
        or any(identity.get('encoder', {}).get(k) != v for k,v in spec.items())
        or (family == 'qwen' and identity.get('encoder', {}).get('input_role') != 'corpus')):
        raise ValueError('Declared embedding producer differs from the frozen candidate.')
    saved = load('embeddings.json')
    if saved is None:
        return None
    checked(saved)
    if (any(saved.get(k) != v for k,v in identity.items())
        or saved.get('signature') != digest_json(identity)):
        raise ValueError('Declared vector manifest identity changed.')
    if family == 'bge' and saved.get('state') != 'COMPLETE':
        return None
    if saved.get('state') not in ('COMPLETE', 'IN_PROGRESS'):
        raise ValueError('Invalid vector manifest state.')
    sources = {p['start']:p for p in inputs['parts']}
    seen, last = set(), -1
    for part in saved['parts']:
        checked(part)
        source = sources.get(part['start'])
        if source is None or part['start'] <= last or part['start'] in seen:
            raise ValueError('Vector manifest coverage/order changed.')
        expected = {'signature':saved['signature'], 'start':source['start'], 'rows':source['rows'],
            'input_sha256':source['sha256'], 'path':f"map-{source['start']:012d}.parquet"}
        if any(part.get(k) != v for k,v in expected.items()) or len(part.get('sha256', '')) != 64:
            raise ValueError('Vector manifest part differs from frozen input order.')
        seen.add(part['start']); last = part['start']
    if saved['state'] == 'COMPLETE' and seen != set(sources):
        raise ValueError('Complete vector manifest has a coverage gap.')
    progress(f"{family.upper()} sealed manifest: {len(saved['parts']):,}/{len(sources):,} parts"
        ' | no per-file Drive verification scan; payload SHA checked on use')
    return view, saved


class VerifiedSubmissionIO:
    """Hash legacy local copies once, then reuse unchanged local file identities."""
    def __init__(self, work_dir, *, api=None, progress=print):
        self.work = Path(work_dir).resolve()
        self.work.mkdir(parents=True, exist_ok=True)
        self.api, self.progress, self.lock = api, progress, threading.RLock()
        directory = self.work / 'io_checkpoints'
        directory.mkdir(exist_ok=True)
        self.con = sqlite3.connect(directory / 'local_proofs.sqlite', check_same_thread=False, timeout=30)
        self.con.execute('PRAGMA journal_mode=WAL')
        self.con.execute('CREATE TABLE IF NOT EXISTS proofs(path TEXT PRIMARY KEY,sha TEXT,stamp TEXT,shape TEXT)')
        self.con.commit()

    def target(self, path):
        path = Path(path)
        if path.is_symlink() or not path.resolve().is_relative_to(self.work):
            raise ValueError('Submission cache target escaped its local workspace.')
        return path

    @staticmethod
    def stamp(path):
        s = path.stat()
        return digest_json([s.st_dev,s.st_ino,s.st_size,s.st_mtime_ns,s.st_ctime_ns])

    def known(self, path, checksum):
        path = self.target(path)
        if not path.is_file():
            return False
        with self.lock:
            row = self.con.execute('SELECT sha,stamp FROM proofs WHERE path=?', [str(path)]).fetchone()
        return row == (checksum, self.stamp(path))

    def record(self, path, checksum, *, shape=None):
        with self.lock:
            self.con.execute('INSERT OR REPLACE INTO proofs VALUES (?,?,?,?)',
                [str(path), checksum, self.stamp(path), None if shape is None else digest_json(list(shape))])
            self.con.commit()

    def prove(self, path, checksum):
        path = self.target(path)
        if self.known(path, checksum):
            return
        before = self.stamp(path)
        verify_file(path, checksum)  # Local SSD only, never rehash the Drive source.
        if before != self.stamp(path):
            raise ValueError('Local cache changed while it was checked.')
        self.record(path, checksum)

    def validate(self, path, checksum, values, dimension):
        with self.lock:
            old = self.con.execute('SELECT sha,stamp,shape FROM proofs WHERE path=?', [str(path)]).fetchone()
        expected = (checksum,self.stamp(path),digest_json(list(values.shape)))
        if values.ndim != 2 or values.shape[1] != dimension:
            raise ValueError('Cached vector dimension changed.')
        if old != expected:
            validate_vectors(values,len(values),dimension)
            self.record(path,checksum,shape=values.shape)

    def relative(self, source):
        if self.api is None:
            return None
        for root in (self.api.data_root,self.api.requested_root):
            try:
                value = Path(source).absolute().relative_to(root).as_posix()
            except ValueError:
                continue
            if '..' not in PurePosixPath(value).parts:
                return value
        return None

    def fetch(self, source, target, checksum, metadata=None):
        target = self.target(target)
        if target.is_file():
            self.prove(target,checksum)
            return False,0
        target.parent.mkdir(parents=True,exist_ok=True)
        stage = target.with_name('.'+uuid.uuid4().hex+'.part')
        try:
            relative = self.relative(source)
            if relative is not None and hasattr(self.api,'payload_metadata'):
                metadata = metadata or self.api.payload_metadata(relative)
                self.api.download(metadata,stage,expected_sha256=checksum)
            else:
                sha = hashlib.sha256()
                with Path(source).open('rb') as src, stage.open('wb') as dst:
                    for chunk in iter(lambda:src.read(1024*1024),b''):
                        sha.update(chunk); dst.write(chunk)
                if sha.hexdigest() != checksum:
                    raise ValueError('Copied source payload checksum changed.')
            os.replace(stage,target)
            self.record(target,checksum)
            return True,target.stat().st_size
        finally:
            stage.unlink(missing_ok=True)

    def stage(self, items, label, *, consume=None, reserve=None, workers=4):
        """A bounded window: consume each window before it can be evicted."""
        items = list(items)
        if not items:
            return
        prepared, missing_bytes = [],0
        for source,target,checksum in items:
            metadata = None
            if not Path(target).is_file():
                relative = self.relative(source)
                if relative is not None and hasattr(self.api,'payload_metadata'):
                    metadata = self.api.payload_metadata(relative)
                    missing_bytes += int(metadata['size'])
            prepared.append((source,target,checksum,metadata))
        visible = len(items)>=32 or any(not Path(item[1]).is_file() for item in prepared)
        if visible:
            self.progress(f'{label}: 0/{len(items):,} files | existing cache retained | missing payload {missing_bytes/2**20:.1f} MiB')
        started, new_bytes, reused = time.monotonic(),0,0
        # Ranking repeatedly gathers a few already-cached blocks. Keep this hot
        # path synchronous; no new threads or API metadata calls are necessary.
        pool = None
        try:
            for offset in range(0,len(prepared),workers):
                window = prepared[offset:offset+workers]
                if reserve is not None:
                    reserve(window)
                if all(Path(item[1]).is_file() for item in window):
                    outcomes = (self.fetch(*item) for item in window)
                else:
                    if pool is None:
                        pool = ThreadPoolExecutor(max_workers=workers)
                    futures = [pool.submit(self.fetch,*item) for item in window]
                    outcomes = (future.result() for future in futures)
                for index,(item,outcome) in enumerate(zip(window,outcomes,strict=True),offset):
                    fresh, size = outcome
                    new_bytes += size; reused += not fresh
                    if consume is not None:
                        consume(index,Path(item[1]))
                done = offset+len(window)
                remaining = max(0,missing_bytes-new_bytes)
                elapsed = time.monotonic()-started
                eta = f' | transfer ETA {remaining/(new_bytes/elapsed):.0f}s' if new_bytes and remaining else ''
                if visible and (done % 32 == 0 or done == len(items)):
                    self.progress(f'{label}: {done:,}/{len(items):,} | reused {reused:,}'
                        f' | new {new_bytes/2**20:.1f} MiB | remaining {remaining/2**20:.1f} MiB{eta}')
        except BaseException:
            if pool is not None:
                pool.shutdown(wait=False,cancel_futures=True)
            raise
        else:
            if pool is not None:
                pool.shutdown(wait=True)


class SubmissionVectorParts(OriginalVectorParts):
    """Same vectors/order as native; parallel API transport and local proof reuse."""
    def __init__(self,*args,io,**kwargs):
        super().__init__(*args,**kwargs)
        self.io = io
        self._bundle_attempted = False

    def map_path(self,index):
        p = self.manifest['parts'][index]
        return self.cache / (digest_json([str(self.root),p['path'],p['sha256']])+'.parquet')

    def block_path(self,ref):
        return self.cache / (digest_json([ref['path'],ref['sha256']])+'.npy')

    def _touch(self,path):
        # Mapping files are small; don't evict them to make room for a vector.
        previous = self.disk.pop(path.name,None)
        if previous:
            self.disk_bytes -= previous[1]
        self.disk[path.name] = path,path.stat().st_size
        self.disk_bytes += self.disk[path.name][1]
        for name,(old,size) in list(self.disk.items()):
            if self.disk_bytes <= self.cache_budget:
                break
            if old.suffix == '.parquet' or old.stem in self.maps or name in getattr(self,'_protected',()):
                continue
            old.unlink(missing_ok=True)
            self.disk_bytes -= size; del self.disk[name]
            self.verified.discard(old.stem)

    def reserve(self,window):
        needed = sum(int(m['size']) if m else 16*2**20 for _,t,_,m in window if not Path(t).is_file())
        protected = {Path(t).name for _,t,_,_ in window}
        self._protected = protected
        for name,(path,size) in list(self.disk.items()):
            if shutil.disk_usage(self.cache).free >= needed+16*2**20:
                return
            if name in protected or path.suffix == '.parquet' or path.stem in self.maps:
                continue
            path.unlink(missing_ok=True); self.disk_bytes -= size; del self.disk[name]
        if shutil.disk_usage(self.cache).free < needed+16*2**20:
            raise RuntimeError('Insufficient Colab scratch space for the next vector window; existing Drive vectors retained.')

    def _block(self,ref):
        name = digest_json([ref['path'],ref['sha256']])
        if name not in self.maps:
            path = self.block_path(ref)
            self.io.fetch(self.data_root/ref['path'],path,ref['sha256'])
            values = np.load(path,allow_pickle=False,mmap_mode='r')
            self.io.validate(path,ref['sha256'],values,self.manifest['dimension'])
            self.maps[name] = values
            if len(self.maps)>4:
                self.maps.popitem(last=False)
        self.maps.move_to_end(name); self._touch(self.block_path(ref))
        return self.maps[name]

    def gather(self,positions):
        positions = np.asarray(positions,np.int64)
        if positions.ndim != 1 or np.any(positions<0) or np.any(positions>=self.manifest['input_count']):
            raise ValueError('Invalid content vector positions.')
        result = np.empty((len(positions),self.manifest['dimension']),np.float32)
        if not len(positions):
            return result
        selected = {}
        for out in np.argsort(positions,kind='stable'):
            index = int(np.searchsorted(self.starts,int(positions[out]),side='right'))-1
            selected.setdefault(index,[]).append((int(out),int(positions[out])-self.starts[index]))
        self.restore_mapping_bundle()
        indexes, groups = list(selected),{}
        def read_map(number,path):
            index = indexes[number]
            part = self.manifest['parts'][index]
            table = pq.read_table(path,columns=['path','sha256','row'])
            if table.num_rows != part['rows']:
                raise ValueError('Content map shape changed.')
            outputs,offsets = zip(*selected[index])
            for out,ref in zip(outputs,table.take(pa.array(offsets,type=pa.int64())).to_pylist(),strict=True):
                relative = PurePosixPath(ref['path'])
                if relative.is_absolute() or '..' in relative.parts or '\\' in ref['path'] or not relative.parts:
                    raise ValueError('Content reference escapes its declared corpus.')
                groups.setdefault((ref['path'],ref['sha256']),[]).append((out,ref['row']))
            self._touch(path)
        self.io.stage([(self.root/self.manifest['parts'][i]['path'],self.map_path(i),self.manifest['parts'][i]['sha256'])
            for i in indexes], 'Mapping cache',consume=read_map)
        self.save_mapping_bundle()
        references = list(groups)
        def read_block(number,path):
            relative, checksum = references[number]
            block = self._block({'path':relative,'sha256':checksum})
            outputs,offsets = zip(*groups[relative,checksum])
            if min(offsets)<0 or max(offsets)>=len(block):
                raise ValueError('Content map row escapes its vector block.')
            result[list(outputs)] = block[list(offsets)]
        self.io.stage([(self.data_root/name,self.block_path({'path':name,'sha256':sha}),sha)
            for name,sha in references], 'Vector cache',consume=read_block,reserve=self.reserve)
        return result

    def bundle_binding(self):
        return {'schema_version':1,'kind':'mapping-bundle-v1','data_folder_id':self.io.api.folder_id,
            'source_manifest_sha256':self.manifest['manifest_sha256']}

    def restore_mapping_bundle(self):
        if self._bundle_attempted or self.io.api is None or not hasattr(self.io.api,'publish_cache_checkpoint'):
            return
        self._bundle_attempted = True
        if all(self.map_path(i).is_file() for i in range(len(self.manifest['parts']))):
            return  # Preserve the old runtime's existing mapping cache.
        binding = self.bundle_binding()
        directory = 'retrieval/mapping_bundles/'+binding['source_manifest_sha256']
        saved = self.io.api.relative_json(directory+'/current.json')
        if saved is None:
            return
        checked(saved)
        if saved.get('binding') != binding or saved.get('parts') != len(self.manifest['parts']):
            raise ValueError('Mapping bundle differs from the current frozen view.')
        if saved.get('path') != 'maps-'+saved['sha256']+'.tar.gz':
            raise ValueError('Mapping bundle path changed.')
        payload = self.io.work/'mapping_bundles'/(saved['sha256']+'.tar.gz')
        self.io.fetch(self.data_root/directory/saved['path'],payload,saved['sha256'])
        expected = {p['path']:(i,p) for i,p in enumerate(self.manifest['parts'])}
        seen, total = set(),0
        with tarfile.open(payload,mode='r|gz') as archive:
            for member in archive:
                if not member.isfile() or member.name not in expected or member.name in seen or member.size>16*2**20:
                    raise ValueError('Unexpected file/link in mapping bundle.')
                total += member.size
                if total>512*2**20:
                    raise ValueError('Mapping bundle exceeds its bounded metadata budget.')
                i,p = expected[member.name]; target = self.map_path(i)
                if target.exists():
                    self.io.prove(target,p['sha256'])
                else:
                    target.parent.mkdir(parents=True,exist_ok=True)
                    stage = target.with_name('.'+uuid.uuid4().hex+'.part')
                    try:
                        src, sha = archive.extractfile(member),hashlib.sha256()
                        with src,stage.open('wb') as dst:
                            for chunk in iter(lambda:src.read(1024*1024),b''):
                                sha.update(chunk); dst.write(chunk)
                        if sha.hexdigest()!=p['sha256']:
                            raise ValueError('Mapping bundle member checksum changed.')
                        os.replace(stage,target); self.io.record(target,p['sha256'])
                    finally:
                        stage.unlink(missing_ok=True)
                seen.add(member.name)
        if seen != set(expected):
            raise ValueError('Mapping bundle has missing parts.')
        self._bundle_saved = True
        self.io.progress(f'Mapping bundle restored: {len(seen):,} parts from one Drive payload')

    def save_mapping_bundle(self):
        if (getattr(self,'_bundle_saved',False) or getattr(self,'_bundle_publish_attempted',False) or self.io.api is None
            or not hasattr(self.io.api,'publish_cache_checkpoint')
            or not all(self.io.known(self.map_path(i),p['sha256']) for i,p in enumerate(self.manifest['parts']))):
            return
        binding = self.bundle_binding()
        directory = 'retrieval/mapping_bundles/'+binding['source_manifest_sha256']
        local = self.io.work/'mapping_bundles'/(binding['source_manifest_sha256']+'.tar.gz')
        local.parent.mkdir(parents=True,exist_ok=True)
        if sum(self.map_path(i).stat().st_size for i in range(len(self.manifest['parts'])))>512*2**20:
            return  # Optional bundle never grows without a metadata bound.
        self._bundle_publish_attempted = True
        with tarfile.open(local,mode='w:gz',compresslevel=1) as archive:
            for i,p in enumerate(self.manifest['parts']):
                path = self.map_path(i)
                info = tarfile.TarInfo(p['path']); info.size = path.stat().st_size
                with path.open('rb') as stream:
                    archive.addfile(info,stream)
        checksum = sha256_file(local)
        saved = seal({'binding':binding,'path':'maps-'+checksum+'.tar.gz','sha256':checksum,
            'parts':len(self.manifest['parts'])})
        try:
            self.io.api.publish_cache_checkpoint(directory+'/current.json',local,saved)
            self._bundle_saved = True
            self.io.progress(f'Mapping bundle SAVED_ON_DRIVE: {saved["parts"]:,} parts; future runtimes restore one payload')
        except LookupPublishUnavailable:
            self.io.progress('Mapping bundle upload deferred; local mapping cache retained')


def index_sample(store,positions,io):
    """Checkpoint the exact deterministic ANN sample, not a second corpus copy."""
    positions = np.asarray(positions,np.int64)
    source = store.manifest['manifest_sha256']
    positions_sha = hashlib.sha256(positions.tobytes()).hexdigest()
    binding = {'schema_version':1,'kind':'index-sample-v1','source_manifest_sha256':source,
        'positions_sha256':positions_sha,'data_folder_id':getattr(io.api,'folder_id',None)}
    directory = 'retrieval/index_samples/'+source+'/'+positions_sha
    local = io.work/'index_samples'/digest_json([source,positions_sha])[:32]
    local.mkdir(parents=True,exist_ok=True)
    marker, payload = local/'sample.json',local/'sample.npy'
    saved = checked(read_json(marker)) if marker.is_file() else None
    if saved is None and io.api is not None and hasattr(io.api,'publish_cache_checkpoint'):
        saved = io.api.relative_json(directory+'/current.json')
        if saved is not None:
            checked(saved)
    if saved is not None:
        if saved.get('binding') != binding or saved.get('rows') != len(positions):
            raise ValueError('ANN sample checkpoint source/positions changed.')
        if saved.get('path') != 'sample-'+saved['sha256']+'.npy':
            raise ValueError('ANN sample checkpoint path changed.')
        io.fetch(store.data_root/directory/saved['path'],payload,saved['sha256'])
        values = np.load(payload,allow_pickle=False)
        io.validate(payload,saved['sha256'],values,store.manifest['dimension'])
        if len(values)!=len(positions):
            raise ValueError('ANN sample row count changed.')
        atomic_json(marker,saved)
        io.progress(f'ANN training sample reused: {len(values):,} vectors; no corpus read')
        return values
    io.progress(f'ANN sample preparation: {len(positions):,} existing vectors; bounded parallel payload reads')
    values = np.ascontiguousarray(store.gather(positions),dtype=np.float32)
    validate_vectors(values,len(positions),store.manifest['dimension'])
    stage = payload.with_suffix('.tmp.npy')
    np.save(stage,values,allow_pickle=False); os.replace(stage,payload)
    checksum = sha256_file(payload); io.record(payload,checksum,shape=values.shape)
    saved = seal({'binding':binding,'path':'sample-'+checksum+'.npy','sha256':checksum,
        'parts':1,'rows':len(positions),'dimension':store.manifest['dimension']})
    atomic_json(marker,saved)
    if io.api is not None and hasattr(io.api,'publish_cache_checkpoint'):
        try:
            io.api.publish_cache_checkpoint(directory+'/current.json',payload,saved)
            io.progress(f'ANN sample SAVED_ON_DRIVE: {len(values):,} vectors; source vectors retained')
        except LookupPublishUnavailable:
            io.progress('ANN sample upload deferred; local sample retained')
    return values

"""Bounded parallel map checks and visible GPU startup; producer files stay pinned."""
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import gzip
from pathlib import Path, PurePosixPath
import shutil
import sqlite3
import sys
import threading
import time

from vietmedbridge.artifacts import (atomic_json, digest_json, local_workspace, publish_file,
    read_json, sha256_file, utc_now, verify_file)
from vietmedbridge.content_embeddings import ContentBlocks as NativeContentBlocks, checked, seal
from vietmedbridge.portable_embeddings import resources_relative
from vietmedbridge.scale_vectors import bound

# Notebook cells embed the pure-stdlib mount helper before this adapter. Module
# imports in local tests load the same helper from scripts, outside producer src.
if 'safe_colab_remount' not in globals():
    from scripts.colab_drive_mount import safe_colab_remount
if 'authenticate_drive_receipts' not in globals():
    from scripts.drive_receipts_api import authenticate_drive_receipts, LookupPublishUnavailable
if 'SubmissionVectorParts' not in globals():
    from scripts.submission_vector_io import (SubmissionVectorParts, VerifiedSubmissionIO,
        read_declared_vector_manifest, index_sample)


class ResumableContentBlocks(NativeContentBlocks):
    """Retain validated receipt lookups on local disk across Drive recovery.

    The native content key, encoder identity and vector payload validation stay
    unchanged. A compressed lookup is an optimization, never a coverage receipt.
    """
    def __init__(self, data_root, encoder, work_dir, *, progress=print, receipt_api=None, inventory=True):
        self.receipt_api = receipt_api
        if receipt_api is not None and Path(data_root).absolute() not in (receipt_api.requested_root, receipt_api.data_root):
            raise ValueError('Receipt API root differs from the embedding data root.')
        self.root = receipt_api.data_root if receipt_api is not None else Path(data_root).resolve()
        self.encoder = encoder
        self.identity = {'encoder': encoder.identity, 'dimension': encoder.dimension,
            'format': 'content-float32-l2-v1'}
        self.directory = self.root / 'model_cache' / digest_json(self.identity)[:24]
        if receipt_api is None:
            self.directory.mkdir(parents=True, exist_ok=True)
        self.progress = progress
        self.binding = {'schema_version': 1, 'data_root': str(self.root), 'identity': self.identity}
        self.lookup = Path(work_dir) / 'receipt_lookups' / digest_json(self.binding)[:16]
        self.lookup.mkdir(parents=True, exist_ok=True)
        self.database = self.lookup / 'lookup.sqlite'
        self.snapshot_binding = (dict(schema_version=2, data_folder_id=receipt_api.folder_id, identity=self.identity)
            if receipt_api is not None and hasattr(receipt_api, 'folder_id') else self.binding)
        self.remote = self.root / 'retrieval/content_lookups' / digest_json(self.snapshot_binding)[:16]
        self.legacy_remote = self.root / 'retrieval/content_lookups' / digest_json(self.binding)[:16]
        self.temporary = local_workspace(work_dir)
        self.local = Path(self.temporary.__enter__())
        self.con = None
        self._snapshot_dirty = False
        self._snapshot_busy = False
        self._next_snapshot_count = 256
        self._receipt_count = 0
        try:
            if not self.database.exists():
                if not inventory:
                    raise FileNotFoundError('No local lookup to save; no receipt scan or model inference was started.')
                self._restore()
            self.con = sqlite3.connect(self.database)
            self.con.execute('PRAGMA cache_size=-32768')
            self.con.execute('PRAGMA synchronous=FULL')
            self.con.execute('CREATE TABLE IF NOT EXISTS metadata(value TEXT NOT NULL)')
            row = self.con.execute('SELECT value FROM metadata').fetchone()
            signature = digest_json(self.binding)
            if row is not None and row[0] != signature:
                raise ValueError('Receipt lookup model/root binding changed.')
            if row is None:
                self.con.execute('INSERT INTO metadata VALUES (?)', [signature])
            self.con.execute('CREATE TABLE IF NOT EXISTS vectors(key TEXT PRIMARY KEY,path TEXT,sha TEXT,row INTEGER)')
            self.con.execute('CREATE TABLE IF NOT EXISTS receipts(name TEXT PRIMARY KEY,digest TEXT NOT NULL)')
            self.con.commit()
            known = {row[0] for row in self.con.execute('SELECT name FROM receipts')}
            self._receipt_count = len(known)
            self._next_snapshot_count = max(256, len(known)*2)
            if not inventory:
                self._snapshot_dirty = True
                self._snapshot()
                return
            self.progress(f'BGE receipt inventory: retained {len(known):,} validated receipts; listing new names')
            markers = (receipt_api.receipts(self.identity) if receipt_api is not None
                else sorted(self.directory.glob('*.done.json')))
            pending = [path for path in markers if path.name not in known]
            self.progress(f'BGE receipt inventory: {len(markers)-len(pending):,}/{len(markers):,} reused'
                f' | {len(pending):,} unread | {"API" if receipt_api is not None else "mount"} transport | commit each receipt')
            self._read_pending(pending, len(markers)-len(pending), len(markers))
            self.progress(f'BGE receipt inventory ready: {len(markers):,} receipts | native vector SHA checked on use')
            if receipt_api is not None:
                if pending:
                    self._snapshot_dirty = True
                if hasattr(receipt_api, 'publish_lookup') and not self._snapshot_dirty:
                    saved = receipt_api.relative_json((self.remote / 'current.json').relative_to(self.root).as_posix())
                    if saved is None:
                        self._snapshot_dirty = True
                    else:
                        checked(saved)
                        if saved.get('binding') != self.snapshot_binding:
                            raise ValueError('Published lookup producer/folder changed.')
                        self._snapshot_dirty = saved['receipts'] < self._receipt_count
                if self._snapshot_dirty:
                    self._snapshot()
                self.progress('BGE API receipt lookup ready; no mount I/O for inventory')
            elif pending or not (self.remote / 'current.json').is_file():
                self._snapshot()
        except BaseException:
            self.close()
            raise

    def _read_pending(self, pending, reused, total):
        # Keep at most two reads in flight: pool.map queued the whole directory
        # and discarded its temporary SQLite lookup after any mount failure.
        readers = 4 if self.receipt_api is not None else 2
        pool = ThreadPoolExecutor(max_workers=readers)
        iterator, futures = iter(pending), []
        count, last_log = 0, time.monotonic()
        def refill():
            while len(futures) < readers:
                path = next(iterator, None)
                if path is None:
                    break
                futures.append((path, pool.submit(lambda p: checked(
                    p.read_json() if self.receipt_api is not None else read_json(p)), path)))
        try:
            refill()
            while futures:
                # Register in native filename order, preserving the first
                # existing vector reference when blocks contain overlapping text.
                path, future = futures.pop(0)
                block = future.result()
                if path.name != 'block-' + digest_json(block['keys']) + '.done.json':
                    raise ValueError('Content receipt name/key binding changed.')
                self.register(block)
                count += 1
                if count % 32 == 0 or time.monotonic()-last_log >= 15 or count == len(pending):
                    self.progress(f'BGE receipt inventory: {reused+count:,}/{total:,}'
                        f' | reused {reused:,}, newly indexed {count:,} | local checkpoint committed')
                    last_log = time.monotonic()
                refill()
        except BaseException:
            pool.shutdown(wait=False, cancel_futures=True)
            raise
        else:
            pool.shutdown(wait=True)

    def register(self, block):
        checked(block)
        marker = 'block-' + digest_json(block['keys']) + '.done.json'
        old = self.con.execute('SELECT digest FROM receipts WHERE name=?', [marker]).fetchone()
        if old is not None and old[0] != block['manifest_sha256']:
            raise ValueError('Immutable content receipt changed in the retained lookup.')
        try:
            # Keys and their receipt commit together; interrupted reads never
            # advance the inventory or erase an earlier committed prefix.
            if self.receipt_api is None:
                super().register(block)
            else:
                if block['identity'] != self.identity or len(block['keys']) != block['rows']:
                    raise ValueError('Content vector producer/key mismatch.')
                relative = PurePosixPath(block['path'])
                if relative.is_absolute() or '..' in relative.parts or '\\' in block['path'] or not relative.parts:
                    raise ValueError('API content reference escapes its data root.')
                # No filesystem resolution through FUSE during API inventory.
                # Native vector readers still resolve the actual path and check
                # its SHA256/shape before consuming the vector payload.
                self.con.executemany('INSERT OR IGNORE INTO vectors VALUES (?,?,?,?)',
                    [(key,block['path'],block['sha256'],i) for i,key in enumerate(block['keys'])])
            self.con.execute('INSERT OR IGNORE INTO receipts VALUES (?,?)', [marker, block['manifest_sha256']])
            self.con.commit()
            if old is None:
                self._receipt_count += 1
                self._snapshot_dirty = True
        except BaseException:
            self.con.rollback()
            raise
        if (self.receipt_api is not None and self._receipt_count >= self._next_snapshot_count
            and hasattr(self.receipt_api, 'publish_lookup')):
            self._next_snapshot_count = max(256, self._receipt_count*2)
            self._snapshot()

    def _restore(self):
        remote = self.remote
        marker = remote / 'current.json'
        if self.receipt_api is None:
            if not marker.is_file():
                return
            saved = checked(read_json(marker))
        else:
            value = self.receipt_api.relative_json(marker.relative_to(self.root).as_posix())
            if value is None and self.legacy_remote != self.remote:
                remote = self.legacy_remote
                value = self.receipt_api.relative_json((remote / 'current.json').relative_to(self.root).as_posix())
            if value is None:
                return
            saved = checked(value)
        if saved.get('binding') != (self.snapshot_binding if remote == self.remote else self.binding):
            raise ValueError('Published receipt lookup producer/root changed.')
        relative = PurePosixPath(saved['path'])
        if relative.is_absolute() or len(relative.parts) != 1 or '\\' in saved['path']:
            raise ValueError('Published receipt snapshot path escapes its directory.')
        self.progress('BGE receipt inventory: restore one compressed lookup; no old receipt JSON scan')
        local = self.local / 'lookup.sqlite.gz'
        if self.receipt_api is None:
            shutil.copyfile(bound(remote, saved['path']), local)
        else:
            source = self.receipt_api.relative((remote / saved['path']).relative_to(self.root).as_posix())
            if source is None:
                raise ValueError('Published API receipt lookup payload is missing.')
            self.receipt_api.download(source, local, expected_sha256=saved['sha256'])
        verify_file(local, saved['sha256'])
        stage = self.lookup / 'restore.sqlite'
        try:
            with gzip.open(local, 'rb') as src, stage.open('wb') as dst:
                shutil.copyfileobj(src, dst, length=1024*1024)
            verify_file(stage, saved['sqlite_sha256'])
            # A shared folder has the same identity on another account even when
            # Colab exposes it through a different MyDrive/shortcut path.
            old_binding = saved.get('lookup_binding', saved['binding'])
            restored = sqlite3.connect(stage)
            try:
                rows = restored.execute('SELECT value FROM metadata').fetchall()
                if rows != [(digest_json(old_binding),)] or old_binding.get('identity') != self.identity:
                    raise ValueError('Restored lookup SQLite binding changed.')
                if restored.execute('SELECT count(*) FROM receipts').fetchone()[0] != saved['receipts']:
                    raise ValueError('Restored lookup receipt count changed.')
                if old_binding != self.binding:
                    restored.execute('UPDATE metadata SET value=?', [digest_json(self.binding)])
                    restored.commit()
            finally:
                restored.close()
            stage.replace(self.database)
        finally:
            stage.unlink(missing_ok=True)

    def _snapshot(self):
        if self._snapshot_busy:
            return
        if self.receipt_api is not None and not hasattr(self.receipt_api, 'publish_lookup'):
            return  # Compatibility with older read-only transports; local prefix remains usable.
        self._snapshot_busy = True
        self.progress('BGE receipt inventory: save compressed lookup for future runtimes')
        copy = self.local / 'lookup-backup.sqlite'
        payload = self.local / 'lookup.sqlite.gz'
        try:
            backup = sqlite3.connect(copy)
            try:
                self.con.backup(backup)
            finally:
                backup.close()
            checksum = sha256_file(copy)
            with copy.open('rb') as src, gzip.open(payload, 'wb', compresslevel=1) as dst:
                shutil.copyfileobj(src, dst, length=1024*1024)
            target = self.remote / ('lookup-' + sha256_file(payload) + '.sqlite.gz')
            record = seal({'binding': self.snapshot_binding, 'lookup_binding': self.binding, 'path': target.name,
                'sha256': sha256_file(payload), 'sqlite_sha256': checksum,
                'receipts': self.con.execute('SELECT count(*) FROM receipts').fetchone()[0],
                'updated_at': utc_now(),
                'scope': 'Receipt lookup only; vector payloads remain subject to native SHA/shape checks'})
            if self.receipt_api is None:
                publish_file(payload, target)
                atomic_json(self.remote / 'current.json', record)
            else:
                self.receipt_api.publish_lookup((self.remote / 'current.json').relative_to(self.root).as_posix(),
                    payload, record)
            self._snapshot_dirty = False
            self.progress(f"BGE receipt lookup SAVED_ON_DRIVE: {record['receipts']:,} receipts; next runtime restores this lookup")
        except (OSError, LookupPublishUnavailable) as error:
            # Optional acceleration only: the local committed database survives,
            # and native reads/writes still surface actual mount/space failures.
            self.progress(f'BGE receipt lookup snapshot deferred: {type(error).__name__}; local checkpoint retained')
        finally:
            self._snapshot_busy = False
            copy.unlink(missing_ok=True)
            payload.unlink(missing_ok=True)

    def close(self):
        if self.con is not None:
            try:
                if self.receipt_api is not None and self._snapshot_dirty:
                    self._snapshot()
            finally:
                self.con.close()
                self.con = None
        self.temporary.__exit__(None, None, None)


def read_bge_maps_parallel(data_root, candidate, inputs, config, *, progress=print):
    """Same BGE identity/map/payload checks as imported_content_manifest, in frozen order."""
    view = Path(data_root) / resources_relative(candidate, config) / 'bge'
    path = view / 'config.json'
    if not path.is_file():
        return view, None
    identity = read_json(path)
    if (identity.get('input_manifest_sha256') != inputs['manifest_sha256']
        or identity.get('input_count') != inputs['input_count']
        or identity.get('format') != 'content-reference-v1'
        or identity.get('dimension') != 1024
        or any(identity.get('encoder', {}).get(k) != v for k, v in config['dense'].items())):
        raise ValueError('Imported BGE view differs from the active frozen corpus/model.')
    signature = digest_json(identity)
    sources = inputs['parts']
    progress(f'BGE mapping check: 0/{len(sources):,} | up to 8 readers | SHA256 preserved')

    def check(source):
        marker = view / f"map-{source['start']:012d}.done.json"
        if not marker.is_file():
            return None
        saved = checked(read_json(marker))
        expected = {'signature': signature, 'start': source['start'], 'rows': source['rows'],
            'input_sha256': source['sha256'], 'path': f"map-{source['start']:012d}.parquet"}
        if any(saved.get(k) != v for k, v in expected.items()):
            raise ValueError('Imported BGE mapping differs from the frozen input order.')
        verify_file(view / saved['path'], saved['sha256'])
        return saved

    ordered = [None] * len(sources)
    found = 0
    pool = ThreadPoolExecutor(max_workers=min(8, max(1, len(sources))))
    try:
        futures = {pool.submit(check, source): i for i, source in enumerate(sources)}
        for count, future in enumerate(as_completed(futures), 1):
            saved = future.result()
            ordered[futures[future]] = saved
            found += saved is not None
            if count % 32 == 0 or count == len(sources):
                progress(f'BGE mapping check: {count:,}/{len(sources):,} | valid maps {found:,}')
    except BaseException:
        # Do not trap the notebook in executor shutdown if Drive readers stall.
        # In-flight work only reads files; it never publishes or changes checkpoints.
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    else:
        pool.shutdown(wait=True)
    parts = [p for p in ordered if p is not None]
    return view, seal(identity | {'signature': signature, 'parts': parts,
        'state': 'COMPLETE' if len(parts) == len(sources) else 'IN_PROGRESS'})


def ensure_submission_mount(data_root, expected_candidate_sha256):
    """Return the matching root through known Drive aliases; JSON only."""
    from vietmedbridge.colab_storage import _disconnected, _drive_mountpoint
    from vietmedbridge.scale_benchmark import load_handoff
    if not isinstance(expected_candidate_sha256, str) or not expected_candidate_sha256:
        raise ValueError('Expected frozen candidate is required before reconnecting.')
    print('SUBMISSION preflight: read frozen handoff JSON; preparation is not repeated', flush=True)
    root = Path(data_root)
    mount = _drive_mountpoint(root)
    # These are observed aliases of this shared project, not a Drive-wide search.
    aliases = [root]
    if mount is not None:
        aliases += [Path('/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge/data'),
            Path('/content/drive/MyDrive/VietMedBridge/data')]
    aliases = list(dict.fromkeys(aliases))
    missing = None
    for attempt in range(2 if mount is not None else 1):
        for location in aliases:
            try:
                _, candidate, _ = load_handoff(location)
            except FileNotFoundError as error:
                missing = error
                continue
            except OSError as error:
                if mount is None or not _disconnected(error, mount):
                    raise
                if attempt:
                    raise
                break
            else:
                if candidate['candidate_manifest_sha256'] != expected_candidate_sha256:
                    raise ValueError('Active candidate changed; refusing to resume another submission corpus.')
                print('SUBMISSION preflight: same frozen candidate verified | DATA_ROOT:', location, flush=True)
                return location
        if mount is not None and not attempt:
            print('SUBMISSION preflight: handoff unavailable; reconnect Drive, retain runtime and local lookup', flush=True)
            safe_colab_remount(mount, readonly=False)
    if mount is None and missing is not None:
        raise missing
    raise FileNotFoundError('Frozen submission handoff not found after reconnect. Checked only: '
        + ', '.join(map(str, aliases)) + '. No data or checkpoint was created.')


def prepare_drive_receipt_lookup(receipt_api, checkout, work_dir, *, progress=print):
    """CPU metadata only, before mapping scans or any neural model allocation."""
    from types import SimpleNamespace
    config = read_json(Path(checkout) / 'configs/retrieval_full.json')
    relative = Path(resources_relative(receipt_api.candidate, config)) / 'bge/config.json'
    header = receipt_api.relative_json(relative.as_posix())
    if header is None:
        progress('BGE API lookup: no existing BGE view; no paid receipt inventory to prepare')
        return
    if (header.get('input_manifest_sha256') != receipt_api.active['index_inputs_manifest_sha256']
        or header.get('dimension') != 1024 or header.get('format') != 'content-reference-v1'
        or any(header.get('encoder', {}).get(k) != v for k,v in config['dense'].items())):
        raise ValueError('API BGE producer/header differs from the active frozen inputs.')
    # A completed, sealed view already carries every map's source/checksum.
    # Do not construct the content-key receipt inventory just to read that view.
    build_run = receipt_api.active.get('build_run')
    inputs = receipt_api.relative_json('processed/'+build_run+'/index_inputs/units.json') if build_run else None
    if inputs is not None:
        if inputs.get('manifest_sha256') != receipt_api.active['index_inputs_manifest_sha256']:
            raise ValueError('Frozen input manifest differs from the active handoff.')
        declared = read_declared_vector_manifest(receipt_api.requested_root,receipt_api.candidate,
            inputs,config,api=receipt_api,progress=progress)
        if declared is not None:
            progress('BGE COMPLETE: skip content receipt inventory and per-map preflight; validate payload on use')
            return
    progress('BGE API lookup preparation: CPU metadata only; no mapping payload scan or model weights')
    encoder = SimpleNamespace(identity=header['encoder'], dimension=header['dimension'])
    lookup = ResumableContentBlocks(receipt_api.requested_root, encoder, work_dir,
        progress=progress, receipt_api=receipt_api)
    lookup.close()
    progress('BGE API receipt lookup ready; continuing native submission checks')


def save_existing_drive_receipt_lookup(receipt_api, checkout, work_dir, *, progress=print):
    """Save an older cell's committed local lookup, without listing receipts or encoding."""
    from types import SimpleNamespace
    config = read_json(Path(checkout) / 'configs/retrieval_full.json')
    relative = Path(resources_relative(receipt_api.candidate, config)) / 'bge/config.json'
    header = receipt_api.relative_json(relative.as_posix())
    if (header is None or header.get('input_manifest_sha256') != receipt_api.active['index_inputs_manifest_sha256']
        or header.get('dimension') != 1024 or header.get('format') != 'content-reference-v1'
        or any(header.get('encoder', {}).get(k) != v for k,v in config['dense'].items())):
        raise ValueError('Saved lookup producer differs from the current frozen BGE view.')
    encoder = SimpleNamespace(identity=header['encoder'], dimension=header['dimension'])
    lookup = ResumableContentBlocks(receipt_api.requested_root, encoder, work_dir,
        progress=progress, receipt_api=receipt_api, inventory=False)
    lookup.close()
    if lookup._snapshot_dirty:
        raise LookupPublishUnavailable('Lookup remains local; Drive save was not confirmed. Keep this runtime.')


class SubmissionProgress:
    def __init__(self, data_root, work_dir, label, *, interval=15, drive_reports=True):
        self.root, self.work, self.label = Path(data_root), Path(work_dir), label
        self.interval, self.main_id = interval, threading.get_ident()
        self.started, self.last_saved = time.monotonic(), 0
        self.phase = 'Entering submission workflow'
        self.binding = {}
        self.drive_reports = drive_reports
        self.stop = threading.Event()

    def location(self):
        frame = sys._current_frames().get(self.main_id)
        location = None
        while frame is not None:
            if frame.f_code.co_name in {'read_json', 'sha256_file'}:
                value = frame.f_locals.get('path')
                if isinstance(value, (str, Path)):
                    location = str(value)
                    break
            if location is None and 'vietmedbridge' in frame.f_code.co_filename:
                location = Path(frame.f_code.co_filename).name + ':' + str(frame.f_lineno)
            frame = frame.f_back
        return location

    def snapshot(self, state='RUNNING'):
        return {'state': state, 'variant': self.label, 'phase': self.phase,
            'elapsed_seconds': round(time.monotonic() - self.started, 1),
            'waiting_at': self.location(), 'updated_at': utc_now(),
            **self.binding,
            'scope': 'Live stage diagnostic; not an embedding completion or submission receipt'}

    def report(self, message):
        self.phase = message
        print(message, flush=True)
        if self.drive_reports and time.monotonic() - self.last_saved >= 15:
            try:
                atomic_json(self.root / 'retrieval/submission_progress.json', self.snapshot())
                self.last_saved = time.monotonic()
            except OSError:
                pass

    def heartbeat(self):
        while not self.stop.wait(self.interval):
            status = self.snapshot()
            print(f"SUBMISSION heartbeat | {status['elapsed_seconds']:.0f}s | {self.phase}"
                f" | waiting at {status['waiting_at']}", flush=True)
            try:
                atomic_json(self.work / 'submission_progress.json', status)
            except OSError:
                pass  # Diagnostics cannot replace or invalidate native checkpoint errors.


@contextmanager
def submission_gpu_progress(data_root, work_dir, *, label='baseline', receipt_api=None):
    """Single-writer notebook adapter; restores every patched binding on exit."""
    from vietmedbridge import (colab_storage, content_embeddings, scale_baseline, submission_workflow,
        full_scale_runtime, cached_strong_runtime, vector_search)
    progress = SubmissionProgress(data_root, work_dir, label, drive_reports=receipt_api is None)
    original_maps = scale_baseline.imported_bge_manifest
    original_blocks = content_embeddings.ContentBlocks
    original_remount = colab_storage._remount
    io = VerifiedSubmissionIO(work_dir,api=receipt_api,progress=progress.report)
    vector_modules = (content_embeddings,scale_baseline,full_scale_runtime,cached_strong_runtime)
    vector_originals = [(module,module.ContentVectorParts) for module in vector_modules]
    content_originals = [(module,module.imported_content_manifest)
        for module in (scale_baseline,cached_strong_runtime)]
    original_content = scale_baseline.imported_content_manifest
    original_build = vector_search.build_faiss_index
    search_originals = [(module,module.search_vectors)
        for module in (vector_search,full_scale_runtime,cached_strong_runtime)]
    original_search = vector_search.search_vectors
    names = ('_submission_context', '_saved_submission', '_release_catalog_staging', '_ensure_primary')
    originals = {name: getattr(submission_workflow, name) for name in names}

    def wrap(name, function):
        def observed(*args, **kwargs):
            progress.report('SUBMISSION stage: ' + name)
            result = function(*args, **kwargs)
            if name == '_submission_context':
                progress.binding = {'build_run': result[3].name,
                    'candidate_manifest_sha256': result[4]['candidate_manifest_sha256'],
                    'input_count': result[5]['input_count']}
            progress.report('SUBMISSION stage complete: ' + name)
            return result
        return observed

    def content_manifest(*args, family):
        result = read_declared_vector_manifest(*args,family=family,api=receipt_api,progress=progress.report)
        if result is not None:
            return result
        if family == 'bge':
            return read_bge_maps_parallel(*args,progress=progress.report)
        return original_content(*args,family=family)

    class SampleStore:
        def __init__(self,store):
            self.store = store
        def __getattr__(self,name):
            return getattr(self.store,name)
        def gather(self,positions):
            return index_sample(self.store,positions,io)

    def build_index(store,*args,**kwargs):
        progress.report('FAISS index: reuse completed shards; checkpoint exact training sample')
        if isinstance(store,SubmissionVectorParts):
            store = SampleStore(store)
        return original_build(store,*args,**kwargs)

    def search(*args,**kwargs):
        progress.report('Vector search: cached corpus vectors; native ANN recall check and query checkpoints')
        return original_search(*args,**kwargs)

    for module,_ in vector_originals:
        module.ContentVectorParts = lambda *a, **k: SubmissionVectorParts(*a,io=io,**k)
    for module,_ in content_originals:
        module.imported_content_manifest = content_manifest
    for module,_ in search_originals:
        module.search_vectors = search
    vector_search.build_faiss_index = build_index
    scale_baseline.imported_bge_manifest = lambda *a, **k: content_manifest(*a,**k,family='bge')
    def resumable_content_blocks(*args, **kwargs):
        data_root = args[0] if args else kwargs.get('data_root')
        api = receipt_api
        if api is not None:
            root = Path(data_root)
            allowed = (Path(api.requested_root), Path(api.data_root))
            if all(root.absolute() != item and root.resolve() != item.resolve() for item in allowed):
                api = None  # Generated document vectors live under output_root, not the source corpus.
        kwargs['progress'], kwargs['receipt_api'] = progress.report, api
        return ResumableContentBlocks(*args, **kwargs)
    content_embeddings.ContentBlocks = resumable_content_blocks
    colab_storage._remount = safe_colab_remount
    for name, function in originals.items():
        setattr(submission_workflow, name, wrap(name, function))
    thread = threading.Thread(target=progress.heartbeat, name='submission-progress', daemon=True)
    progress.report('SUBMISSION startup | live progress every 15 seconds | same candidate and checkpoints')
    thread.start()
    state = 'STOPPED'
    try:
        yield progress
        state = 'CALL_FINISHED'
    finally:
        progress.stop.set()
        thread.join(timeout=1)
        scale_baseline.imported_bge_manifest = original_maps
        content_embeddings.ContentBlocks = original_blocks
        colab_storage._remount = original_remount
        for module,function in vector_originals:
            module.ContentVectorParts = function
        for module,function in content_originals:
            module.imported_content_manifest = function
        for module,function in search_originals:
            module.search_vectors = function
        vector_search.build_faiss_index = original_build
        for name, function in originals.items():
            setattr(submission_workflow, name, function)
        status = progress.snapshot(state)
        try:
            atomic_json(Path(work_dir) / 'submission_progress.json', status)
        except OSError:
            pass
        if receipt_api is None or state == 'CALL_FINISHED':
            try:
                atomic_json(Path(data_root) / 'retrieval/submission_progress.json', status)
            except OSError:
                pass


from vietmedbridge.submission_workflow import run_current_baseline
BASELINE = None
if GPU_SESSION:
    DATA_ROOT = ensure_submission_mount(DATA_ROOT, PREPARATION["candidate_manifest_sha256"])
    DRIVE_RECEIPTS = authenticate_drive_receipts(DATA_ROOT, PREPARATION["candidate_manifest_sha256"])
    with submission_gpu_progress(DATA_ROOT, WORK_DIR, label="baseline", receipt_api=DRIVE_RECEIPTS) as receipt_progress:
        prepare_drive_receipt_lookup(DRIVE_RECEIPTS, CHECKOUT, WORK_DIR, progress=receipt_progress.report)
        BASELINE = run_current_baseline(DATA_ROOT,CHECKOUT,WORK_DIR,code_commit=COMMIT)
    print(json.dumps(BASELINE["ready"],ensure_ascii=False,indent=2),flush=True)
    if BASELINE["ready"]["state"] != "READY_FOR_MANUAL_UPLOAD":
        raise RuntimeError("Baseline chưa đủ query/index: Run all cùng cấu hình để resume. Chưa nạp Qwen reranker.")
    print("ĐÃ CÓ ZIP ĐẦU TIÊN trên Drive. Cell cuối hiển thị vị trí bản này trên Drive.",flush=True)
else:
    print("GPU phase skipped on CPU.")


In [ ]:
#@title 4. Bản nâng cấp: dịch + expansion + document dense + Qwen rerank + ZIP (GPU)
"""Repair a nonempty, unmounted Colab Drive directory without deleting files.

Pure standard library so notebook setup can embed this before pip/package imports.
Kernel mount state must positively exclude this directory and any nested mounts
before an atomic rename preserves the complete local directory under /content.
"""
import os
from pathlib import Path
import re
from uuid import uuid4


def preserve_unmounted_drive_directory(mount, *, content_root='/content',
                                       mountinfo_file='/proc/self/mountinfo'):
    target, parent = Path(mount).absolute(), Path(content_root).resolve()
    if target.name != 'drive' or target.parent.resolve() != parent:
        raise ValueError('Drive recovery may preserve only the named content_root/drive directory.')
    if target.is_symlink():
        raise RuntimeError('Drive mountpoint is a symlink; refusing to move it.')
    try:
        entries = Path(mountinfo_file).read_text(encoding='utf-8').splitlines()
    except OSError as error:
        raise RuntimeError('Cannot verify kernel mount state; no Drive directory was moved.') from error
    if not entries:
        raise RuntimeError('Empty kernel mount information; no Drive directory was moved.')
    for line in entries:
        fields = line.split()
        if len(fields) < 10 or '-' not in fields[6:]:
            raise RuntimeError('Invalid kernel mount information; no Drive directory was moved.')
        name = re.sub(r'\\([0-7]{3})', lambda match: chr(int(match[1], 8)), fields[4])
        point = Path(name).absolute()
        if point == target or point.is_relative_to(target):
            raise RuntimeError(f'Kernel mount still active at {point}; refusing to move Drive data.')
    if os.path.ismount(target):
        raise RuntimeError('Drive is still a mountpoint; refusing to move it.')
    if not target.exists():
        return None
    if not target.is_dir():
        raise RuntimeError('Drive mountpoint is a file; refusing to move it.')
    if next(target.iterdir(), None) is None:
        return None
    backup = parent / ('vmb_drive_unmounted_' + uuid4().hex)
    if backup.exists() or backup.is_symlink():
        raise RuntimeError('Preservation path already exists; no Drive directory was moved.')
    # Same-parent rename retains every nested file and never follows child links.
    # Only an unmounted local directory is moved; original remote data is untouched.
    target.rename(backup)
    target.mkdir()
    print('Local unmounted Drive directory preserved at:', backup, flush=True)
    return backup


def safe_colab_mount(mount='/content/drive', *, force_remount=False, readonly=False,
                     drive_api=None, content_root='/content', mountinfo_file='/proc/self/mountinfo'):
    if drive_api is None:
        from google.colab import drive as drive_api
    try:
        drive_api.mount(str(mount), force_remount=force_remount, readonly=readonly)
        return None
    except ValueError as error:
        if 'Mountpoint must not already contain files' not in str(error):
            raise
    backup = preserve_unmounted_drive_directory(mount, content_root=content_root,
        mountinfo_file=mountinfo_file)
    drive_api.mount(str(mount), force_remount=True, readonly=readonly)
    return backup


def safe_colab_remount(mount, *, readonly=False, drive_api=None,
                       content_root='/content', mountinfo_file='/proc/self/mountinfo'):
    if drive_api is None:
        from google.colab import drive as drive_api
    try:
        drive_api.flush_and_unmount()
    except Exception as error:
        print('Drive flush/unmount did not complete:', type(error).__name__, flush=True)
    return safe_colab_mount(mount, force_remount=True, readonly=readonly, drive_api=drive_api,
        content_root=content_root, mountinfo_file=mountinfo_file)

"""API receipt reads and narrowly scoped lookup checkpoints, without Drive FUSE."""
import copy
import hashlib
import json
from pathlib import Path, PurePosixPath
import re
import threading
import time


class LookupPublishUnavailable(RuntimeError):
    """An optional lookup upload failed; paid vector receipts remain untouched."""


class RemoteReceipt:
    def __init__(self, source, metadata):
        self.source, self.metadata = source, metadata
        self.name = metadata['name']

    def read_json(self):
        return self.source.read_json(self.metadata)


class DriveReceiptSource:
    endpoint = 'https://www.googleapis.com/drive/v3/files'

    def __init__(self, session_factory, data_root, canonical_root, *,
                 data_folder_id='1Ez1S9jk6FO_puxC8r2vPSc0LJbuOkkdj', report=print):
        if not re.fullmatch(r'[A-Za-z0-9_-]+', data_folder_id):
            raise ValueError('Invalid Drive data folder ID.')
        self.session_factory, self.sessions = session_factory, threading.local()
        self.requested_root, self.data_root = Path(data_root).absolute(), Path(canonical_root)
        self.folder_id, self.report, self.folders = data_folder_id, report, {}
        self.payload_directories = {}

    def directory_files(self, relative_directory, *, refresh=False):
        """One paginated metadata listing, rather than one lookup per payload."""
        relative = PurePosixPath(relative_directory)
        if relative.is_absolute() or '..' in relative.parts or '\\' in relative_directory or not relative.parts:
            raise ValueError('Invalid payload directory.')
        if not refresh and relative_directory in self.payload_directories:
            return self.payload_directories[relative_directory]
        parent = self.folder_id
        for part in relative.parts:
            if (parent, part) not in self.folders:
                value = self.child(parent, part, folder=True)
                if value is None:
                    return {}
                self.folders[parent, part] = value['id']
            parent = self.folders[parent, part]
        result = {}
        for value in self._list(f"'{parent}' in parents and trashed=false"):
            if value['name'] in result:
                raise ValueError('Duplicate payload names in Drive directory.')
            result[value['name']] = value
        self.payload_directories[relative_directory] = result
        return result

    def payload_metadata(self, relative_path):
        relative = PurePosixPath(relative_path)
        if relative.is_absolute() or '..' in relative.parts or '\\' in relative_path or len(relative.parts) < 2:
            raise ValueError('Invalid immutable payload path.')
        directory = relative.parent.as_posix()
        result = self.directory_files(directory).get(relative.name)
        if result is None:
            result = self.directory_files(directory, refresh=True).get(relative.name)
        if result is None or 'size' not in result:
            raise ValueError('Expected immutable payload is missing from Drive.')
        return result

    def _session(self):
        if not hasattr(self.sessions, 'session'):
            self.sessions.session = self.session_factory()
        return self.sessions.session

    def _get(self, url, params, *, stream=False):
        import requests
        session = self._session()
        for attempt in range(4):
            try:
                response = session.get(url, params=params, stream=stream, timeout=(10,60))
            except requests.RequestException:
                if attempt == 3:
                    raise
            else:
                if response.status_code not in (429,500,502,503,504) or attempt == 3:
                    response.raise_for_status()
                    return response
                response.close()
            self.report(f'Drive receipt API: transient request failure; retry {attempt+1}/3')
            time.sleep(2**attempt)

    def _list(self, query):
        token, seen = None, set()
        while True:
            params = {'q': query, 'pageSize': 1000, 'supportsAllDrives': 'true',
                'includeItemsFromAllDrives': 'true',
                'fields': 'nextPageToken,incompleteSearch,files(id,name,mimeType,size,md5Checksum)'}
            if token:
                params['pageToken'] = token
            with self._get(self.endpoint, params) as response:
                page = response.json()
            if page.get('incompleteSearch'):
                raise ValueError('Drive receipt listing is incomplete; refusing partial inventory.')
            yield from page.get('files', [])
            token = page.get('nextPageToken')
            if not token:
                return
            if token in seen:
                raise ValueError('Drive receipt pagination repeated; inventory is incomplete.')
            seen.add(token)

    def child(self, parent, name, *, folder=False):
        safe = name.replace('\\', '\\\\').replace("'", "\\'")
        query = f"'{parent}' in parents and trashed=false and name='{safe}'"
        if folder:
            query += " and mimeType='application/vnd.google-apps.folder'"
        values = list(self._list(query))
        if len(values) > 1:
            raise ValueError('Duplicate Drive path names; refusing ambiguous receipt source.')
        return values[0] if values else None

    def relative(self, name):
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or '\\' in name or not relative.parts:
            raise ValueError('Invalid relative Drive receipt path.')
        parent = self.folder_id
        for part in relative.parts[:-1]:
            key = parent, part
            if key not in self.folders:
                value = self.child(parent, part, folder=True)
                if value is None:
                    return None
                self.folders[key] = value['id']
            parent = self.folders[key]
        return self.child(parent, relative.name)

    def download(self, metadata, target, *, expected_sha256=None, max_bytes=None):
        size = int(metadata['size'])
        if max_bytes is not None and size > max_bytes:
            raise ValueError('Receipt exceeds the bounded JSON size.')
        sha, md5, count = hashlib.sha256(), hashlib.md5(), 0
        with self._get(self.endpoint+'/'+metadata['id'], {'alt':'media','supportsAllDrives':'true'}, stream=True) as response:
            with Path(target).open('wb') as output:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    count += len(chunk)
                    if count > size or (max_bytes is not None and count > max_bytes):
                        raise ValueError('Drive receipt payload exceeds its declared size.')
                    sha.update(chunk)
                    md5.update(chunk)
                    output.write(chunk)
        if (count != size or (metadata.get('md5Checksum') and md5.hexdigest() != metadata['md5Checksum'])
            or (expected_sha256 and sha.hexdigest() != expected_sha256)):
            raise ValueError('Drive receipt size/checksum mismatch.')

    def read_json(self, metadata):
        import tempfile
        with tempfile.TemporaryDirectory(prefix='vmb-api-json-') as directory:
            path = Path(directory) / 'receipt.json'
            self.download(metadata, path, max_bytes=16*1024*1024)
            return json.loads(path.read_text(encoding='utf-8'))

    def relative_json(self, path):
        metadata = self.relative(path)
        return None if metadata is None else self.read_json(metadata)

    def verify_candidate(self, expected):
        active = self.relative_json('active_data_candidate.json')
        if active is None or active.get('candidate_manifest_sha256') != expected:
            raise ValueError('Drive API data folder differs from the expected frozen candidate.')
        candidate = self.relative_json('processed/'+active['build_run']+'/'+active['candidate_name'])
        from vietmedbridge.artifacts import digest_json
        if (candidate is None or candidate.get('candidate_manifest_sha256') != expected
            or digest_json({k:v for k,v in candidate.items() if k != 'candidate_manifest_sha256'}) != expected):
            raise ValueError('Drive API candidate digest changed.')
        self.active, self.candidate = active, candidate

    def receipts(self, identity):
        from vietmedbridge.artifacts import digest_json
        # Resolve the folder itself without relying on any receipt being present.
        cache = self.child(self.folder_id, 'model_cache', folder=True)
        folder = self.child(cache['id'], digest_json(identity)[:24], folder=True) if cache else None
        if folder is None:
            return []
        result, names, payloads = [], set(), {}
        # Drive name-contains is prefix matching, not a suffix predicate. List
        # the complete producer folder then filter receipt names locally.
        query = f"'{folder['id']}' in parents and trashed=false"
        for value in self._list(query):
            if value['name'] in payloads:
                raise ValueError('Duplicate names in content cache directory.')
            payloads[value['name']] = value
            if not value['name'].endswith('.done.json'):
                continue
            if (not re.fullmatch(r'block-[0-9a-f]{64}\.done\.json', value['name'])
                or value['name'] in names or 'size' not in value):
                raise ValueError('Invalid or duplicate API receipt filename.')
            names.add(value['name'])
            result.append(RemoteReceipt(self, value))
        self.payload_directories['model_cache/'+digest_json(identity)[:24]] = payloads
        self.report(f'Drive receipt API: complete listing, {len(result):,} receipts; no mount reads')
        return sorted(result, key=lambda value: value.name)

    def publish_cache_checkpoint(self, relative_marker, payload, marker):
        """Only mapping bundles and exact ANN training samples may be published here."""
        from vietmedbridge.artifacts import digest_json
        binding = marker.get('binding', {})
        source = binding.get('source_manifest_sha256', '')
        kind = binding.get('kind')
        if kind == 'mapping-bundle-v1':
            directory, prefix, suffix = 'retrieval/mapping_bundles/'+source, 'maps-', '.tar.gz'
        elif kind == 'index-sample-v1' and re.fullmatch(r'[0-9a-f]{64}', binding.get('positions_sha256', '')):
            directory = 'retrieval/index_samples/'+source+'/'+binding['positions_sha256']
            prefix, suffix = 'sample-', '.npy'
        else:
            raise ValueError('Unsupported cache checkpoint kind.')
        if (binding.get('data_folder_id') != self.folder_id or binding.get('schema_version') != 1
            or not re.fullmatch(r'[0-9a-f]{64}', source)
            or relative_marker != directory+'/current.json'
            or not re.fullmatch(r'[0-9a-f]{64}', marker.get('sha256', ''))
            or marker.get('path') != prefix+marker['sha256']+suffix
            or marker.get('manifest_sha256') != digest_json({k:v for k,v in marker.items() if k != 'manifest_sha256'})):
            raise ValueError('Cache checkpoint escapes its declared source/folder binding.')
        sha, md5 = hashlib.sha256(), hashlib.md5()
        with Path(payload).open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024*1024), b''):
                sha.update(chunk); md5.update(chunk)
        if sha.hexdigest() != marker['sha256']:
            raise ValueError('Cache checkpoint payload changed before upload.')
        try:
            return self._publish_lookup_files(directory, Path(payload), marker, md5.hexdigest())
        except ValueError:
            raise
        except Exception as error:
            raise LookupPublishUnavailable(type(error).__name__) from error

    def publish_lookup(self, relative_marker, payload, marker):
        """Upload a checksum-addressed lookup, then its pointer. Never write vector/source files."""
        from vietmedbridge.artifacts import digest_json
        body = {k:v for k,v in marker.items() if k != 'manifest_sha256'}
        binding = marker.get('binding', {})
        expected_directory = 'retrieval/content_lookups/' + digest_json(binding)[:16]
        if (relative_marker != expected_directory + '/current.json'
            or binding.get('schema_version') != 2 or binding.get('data_folder_id') != self.folder_id
            or marker.get('manifest_sha256') != digest_json(body)
            or marker.get('path') != 'lookup-' + marker.get('sha256', '') + '.sqlite.gz'
            or not re.fullmatch(r'lookup-[0-9a-f]{64}\.sqlite\.gz', marker.get('path', ''))):
            raise ValueError('Lookup upload is outside its bound checkpoint directory.')
        sha, md5 = hashlib.sha256(), hashlib.md5()
        with Path(payload).open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024*1024), b''):
                sha.update(chunk)
                md5.update(chunk)
        if sha.hexdigest() != marker['sha256']:
            raise ValueError('Lookup payload changed before upload.')
        try:
            return self._publish_lookup_files(expected_directory, Path(payload), marker, md5.hexdigest())
        except ValueError:
            raise
        except Exception as error:
            # Do not include provider messages/URLs: resumable upload URLs are private.
            raise LookupPublishUnavailable(type(error).__name__) from error

    def _publish_lookup_files(self, directory, payload, marker, md5):
        from vietmedbridge.artifacts import digest_json
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload
        import io
        service = build('drive', 'v3', credentials=copy.copy(self._session().credentials), cache_discovery=False)
        files = service.files()
        fields = 'id,name,mimeType,size,md5Checksum'

        def new_id():
            return files.generateIds(count=1, space='drive', type='files').execute(num_retries=3)['ids'][0]

        def ensure_folder(parent, name):
            value = self.child(parent, name, folder=True)
            if value is None:
                identifier = new_id()
                try:
                    value = files.create(body={'id':identifier, 'name':name, 'parents':[parent],
                        'mimeType':'application/vnd.google-apps.folder'}, fields=fields,
                        supportsAllDrives=True).execute(num_retries=3)
                except Exception:
                    # A lost response may follow a successful create. Reconcile the
                    # preallocated ID; never retry a folder with a different ID.
                    value = files.get(fileId=identifier, fields=fields, supportsAllDrives=True).execute(num_retries=3)
                    if value.get('name') != name or value.get('mimeType') != 'application/vnd.google-apps.folder':
                        raise ValueError('Created lookup folder identity changed.')
            self.folders[parent, name] = value['id']
            return value['id']

        parent = self.folder_id
        for name in PurePosixPath(directory).parts:
            parent = ensure_folder(parent, name)
        current = self.child(parent, 'current.json')
        if current is not None:
            existing = self.read_json(current)
            if (existing.get('manifest_sha256') != digest_json(
                    {k:v for k,v in existing.items() if k != 'manifest_sha256'})
                or existing.get('binding') != marker['binding']):
                raise ValueError('Existing lookup checkpoint binding/digest changed.')
            count_key = 'receipts' if 'receipts' in marker else 'parts'
            if existing[count_key] > marker[count_key]:
                return existing  # An older local prefix must not replace a newer saved lookup.

        def upload(media, name, existing=None):
            if existing is None:
                request = files.create(body={'id':new_id(), 'name':name, 'parents':[parent]},
                    media_body=media, fields=fields, supportsAllDrives=True)
            else:
                request = files.update(fileId=existing['id'], media_body=media,
                    fields=fields, supportsAllDrives=True)
            result = None
            while result is None:
                _, result = request.next_chunk(num_retries=3)
            # Query provider metadata independently before publishing a pointer.
            return files.get(fileId=result['id'], fields=fields, supportsAllDrives=True).execute(num_retries=3)

        saved = self.child(parent, marker['path'])
        if saved is None:
            mime = 'application/gzip' if marker['path'].endswith('.gz') else 'application/octet-stream'
            saved = upload(MediaFileUpload(str(payload), mimetype=mime,
                chunksize=8*1024*1024, resumable=True), marker['path'])
        if int(saved.get('size', -1)) != payload.stat().st_size or saved.get('md5Checksum') != md5:
            raise ValueError('Uploaded lookup payload size/checksum mismatch; pointer not advanced.')
        data = json.dumps(marker, ensure_ascii=False, sort_keys=True).encode('utf-8')
        saved_marker = upload(MediaIoBaseUpload(io.BytesIO(data), mimetype='application/json',
            chunksize=256*1024, resumable=True), 'current.json', current)
        if int(saved_marker.get('size', -1)) != len(data) or saved_marker.get('md5Checksum') != hashlib.md5(data).hexdigest():
            raise ValueError('Uploaded lookup pointer size/checksum mismatch.')
        return marker


def authenticate_drive_receipts(data_root, expected_candidate_sha256):
    """Authenticate on the main Colab thread; never print/save credential tokens."""
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    root = Path(data_root).resolve()
    print('Drive receipts: authenticate API; receipts read-only, lookup checkpoints saved in retrieval/content_lookups', flush=True)
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    client = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data_root, root)
    client.verify_candidate(expected_candidate_sha256)
    print('Drive receipts: API source verified against the same frozen candidate', flush=True)
    return client

"""Cache-aware submission I/O; immutable producer/model source files stay pinned."""
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
import hashlib
import os
from pathlib import Path, PurePosixPath
import shutil
import sqlite3
import tarfile
import threading
import time
import uuid

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from vietmedbridge.artifacts import atomic_json, digest_json, read_json, sha256_file, verify_file
from vietmedbridge.content_embeddings import ContentVectorParts as OriginalVectorParts, checked, seal
from vietmedbridge.embeddings import validate_vectors
from vietmedbridge.portable_embeddings import resources_relative

if 'LookupPublishUnavailable' not in globals():
    from scripts.drive_receipts_api import LookupPublishUnavailable


def read_declared_vector_manifest(data_root, candidate, inputs, config, *, family='bge', api=None, progress=print):
    """Use the sealed manifest, validating actual payloads only when consumed.

    None means the caller must discover partial BGE receipts through the native
    fallback. A declared incomplete Qwen view can only defer that optional leg.
    """
    if family not in ('bge', 'qwen'):
        raise ValueError('Unknown embedding family.')
    checked(inputs)
    view = Path(data_root) / resources_relative(candidate, config) / family
    def load(name):
        if api is not None:
            return api.relative_json((view / name).relative_to(Path(data_root)).as_posix())
        path = view / name
        return read_json(path) if path.is_file() else None
    identity = load('config.json')
    if identity is None:
        return None
    spec = config['dense' if family == 'bge' else 'second_dense']
    if (identity.get('input_manifest_sha256') != inputs['manifest_sha256']
        or identity.get('input_count') != inputs['input_count']
        or identity.get('format') != 'content-reference-v1'
        or identity.get('dimension') != (1024 if family == 'bge' else 4096)
        or any(identity.get('encoder', {}).get(k) != v for k,v in spec.items())
        or (family == 'qwen' and identity.get('encoder', {}).get('input_role') != 'corpus')):
        raise ValueError('Declared embedding producer differs from the frozen candidate.')
    saved = load('embeddings.json')
    if saved is None:
        return None
    checked(saved)
    if (any(saved.get(k) != v for k,v in identity.items())
        or saved.get('signature') != digest_json(identity)):
        raise ValueError('Declared vector manifest identity changed.')
    if family == 'bge' and saved.get('state') != 'COMPLETE':
        return None
    if saved.get('state') not in ('COMPLETE', 'IN_PROGRESS'):
        raise ValueError('Invalid vector manifest state.')
    sources = {p['start']:p for p in inputs['parts']}
    seen, last = set(), -1
    for part in saved['parts']:
        checked(part)
        source = sources.get(part['start'])
        if source is None or part['start'] <= last or part['start'] in seen:
            raise ValueError('Vector manifest coverage/order changed.')
        expected = {'signature':saved['signature'], 'start':source['start'], 'rows':source['rows'],
            'input_sha256':source['sha256'], 'path':f"map-{source['start']:012d}.parquet"}
        if any(part.get(k) != v for k,v in expected.items()) or len(part.get('sha256', '')) != 64:
            raise ValueError('Vector manifest part differs from frozen input order.')
        seen.add(part['start']); last = part['start']
    if saved['state'] == 'COMPLETE' and seen != set(sources):
        raise ValueError('Complete vector manifest has a coverage gap.')
    progress(f"{family.upper()} sealed manifest: {len(saved['parts']):,}/{len(sources):,} parts"
        ' | no per-file Drive verification scan; payload SHA checked on use')
    return view, saved


class VerifiedSubmissionIO:
    """Hash legacy local copies once, then reuse unchanged local file identities."""
    def __init__(self, work_dir, *, api=None, progress=print):
        self.work = Path(work_dir).resolve()
        self.work.mkdir(parents=True, exist_ok=True)
        self.api, self.progress, self.lock = api, progress, threading.RLock()
        directory = self.work / 'io_checkpoints'
        directory.mkdir(exist_ok=True)
        self.con = sqlite3.connect(directory / 'local_proofs.sqlite', check_same_thread=False, timeout=30)
        self.con.execute('PRAGMA journal_mode=WAL')
        self.con.execute('CREATE TABLE IF NOT EXISTS proofs(path TEXT PRIMARY KEY,sha TEXT,stamp TEXT,shape TEXT)')
        self.con.commit()

    def target(self, path):
        path = Path(path)
        if path.is_symlink() or not path.resolve().is_relative_to(self.work):
            raise ValueError('Submission cache target escaped its local workspace.')
        return path

    @staticmethod
    def stamp(path):
        s = path.stat()
        return digest_json([s.st_dev,s.st_ino,s.st_size,s.st_mtime_ns,s.st_ctime_ns])

    def known(self, path, checksum):
        path = self.target(path)
        if not path.is_file():
            return False
        with self.lock:
            row = self.con.execute('SELECT sha,stamp FROM proofs WHERE path=?', [str(path)]).fetchone()
        return row == (checksum, self.stamp(path))

    def record(self, path, checksum, *, shape=None):
        with self.lock:
            self.con.execute('INSERT OR REPLACE INTO proofs VALUES (?,?,?,?)',
                [str(path), checksum, self.stamp(path), None if shape is None else digest_json(list(shape))])
            self.con.commit()

    def prove(self, path, checksum):
        path = self.target(path)
        if self.known(path, checksum):
            return
        before = self.stamp(path)
        verify_file(path, checksum)  # Local SSD only, never rehash the Drive source.
        if before != self.stamp(path):
            raise ValueError('Local cache changed while it was checked.')
        self.record(path, checksum)

    def validate(self, path, checksum, values, dimension):
        with self.lock:
            old = self.con.execute('SELECT sha,stamp,shape FROM proofs WHERE path=?', [str(path)]).fetchone()
        expected = (checksum,self.stamp(path),digest_json(list(values.shape)))
        if values.ndim != 2 or values.shape[1] != dimension:
            raise ValueError('Cached vector dimension changed.')
        if old != expected:
            validate_vectors(values,len(values),dimension)
            self.record(path,checksum,shape=values.shape)

    def relative(self, source):
        if self.api is None:
            return None
        for root in (self.api.data_root,self.api.requested_root):
            try:
                value = Path(source).absolute().relative_to(root).as_posix()
            except ValueError:
                continue
            if '..' not in PurePosixPath(value).parts:
                return value
        return None

    def fetch(self, source, target, checksum, metadata=None):
        target = self.target(target)
        if target.is_file():
            self.prove(target,checksum)
            return False,0
        target.parent.mkdir(parents=True,exist_ok=True)
        stage = target.with_name('.'+uuid.uuid4().hex+'.part')
        try:
            relative = self.relative(source)
            if relative is not None and hasattr(self.api,'payload_metadata'):
                metadata = metadata or self.api.payload_metadata(relative)
                self.api.download(metadata,stage,expected_sha256=checksum)
            else:
                sha = hashlib.sha256()
                with Path(source).open('rb') as src, stage.open('wb') as dst:
                    for chunk in iter(lambda:src.read(1024*1024),b''):
                        sha.update(chunk); dst.write(chunk)
                if sha.hexdigest() != checksum:
                    raise ValueError('Copied source payload checksum changed.')
            os.replace(stage,target)
            self.record(target,checksum)
            return True,target.stat().st_size
        finally:
            stage.unlink(missing_ok=True)

    def stage(self, items, label, *, consume=None, reserve=None, workers=4):
        """A bounded window: consume each window before it can be evicted."""
        items = list(items)
        if not items:
            return
        prepared, missing_bytes = [],0
        for source,target,checksum in items:
            metadata = None
            if not Path(target).is_file():
                relative = self.relative(source)
                if relative is not None and hasattr(self.api,'payload_metadata'):
                    metadata = self.api.payload_metadata(relative)
                    missing_bytes += int(metadata['size'])
            prepared.append((source,target,checksum,metadata))
        visible = len(items)>=32 or any(not Path(item[1]).is_file() for item in prepared)
        if visible:
            self.progress(f'{label}: 0/{len(items):,} files | existing cache retained | missing payload {missing_bytes/2**20:.1f} MiB')
        started, new_bytes, reused = time.monotonic(),0,0
        # Ranking repeatedly gathers a few already-cached blocks. Keep this hot
        # path synchronous; no new threads or API metadata calls are necessary.
        pool = None
        try:
            for offset in range(0,len(prepared),workers):
                window = prepared[offset:offset+workers]
                if reserve is not None:
                    reserve(window)
                if all(Path(item[1]).is_file() for item in window):
                    outcomes = (self.fetch(*item) for item in window)
                else:
                    if pool is None:
                        pool = ThreadPoolExecutor(max_workers=workers)
                    futures = [pool.submit(self.fetch,*item) for item in window]
                    outcomes = (future.result() for future in futures)
                for index,(item,outcome) in enumerate(zip(window,outcomes,strict=True),offset):
                    fresh, size = outcome
                    new_bytes += size; reused += not fresh
                    if consume is not None:
                        consume(index,Path(item[1]))
                done = offset+len(window)
                remaining = max(0,missing_bytes-new_bytes)
                elapsed = time.monotonic()-started
                eta = f' | transfer ETA {remaining/(new_bytes/elapsed):.0f}s' if new_bytes and remaining else ''
                if visible and (done % 32 == 0 or done == len(items)):
                    self.progress(f'{label}: {done:,}/{len(items):,} | reused {reused:,}'
                        f' | new {new_bytes/2**20:.1f} MiB | remaining {remaining/2**20:.1f} MiB{eta}')
        except BaseException:
            if pool is not None:
                pool.shutdown(wait=False,cancel_futures=True)
            raise
        else:
            if pool is not None:
                pool.shutdown(wait=True)


class SubmissionVectorParts(OriginalVectorParts):
    """Same vectors/order as native; parallel API transport and local proof reuse."""
    def __init__(self,*args,io,**kwargs):
        super().__init__(*args,**kwargs)
        self.io = io
        self._bundle_attempted = False

    def map_path(self,index):
        p = self.manifest['parts'][index]
        return self.cache / (digest_json([str(self.root),p['path'],p['sha256']])+'.parquet')

    def block_path(self,ref):
        return self.cache / (digest_json([ref['path'],ref['sha256']])+'.npy')

    def _touch(self,path):
        # Mapping files are small; don't evict them to make room for a vector.
        previous = self.disk.pop(path.name,None)
        if previous:
            self.disk_bytes -= previous[1]
        self.disk[path.name] = path,path.stat().st_size
        self.disk_bytes += self.disk[path.name][1]
        for name,(old,size) in list(self.disk.items()):
            if self.disk_bytes <= self.cache_budget:
                break
            if old.suffix == '.parquet' or old.stem in self.maps or name in getattr(self,'_protected',()):
                continue
            old.unlink(missing_ok=True)
            self.disk_bytes -= size; del self.disk[name]
            self.verified.discard(old.stem)

    def reserve(self,window):
        needed = sum(int(m['size']) if m else 16*2**20 for _,t,_,m in window if not Path(t).is_file())
        protected = {Path(t).name for _,t,_,_ in window}
        self._protected = protected
        for name,(path,size) in list(self.disk.items()):
            if shutil.disk_usage(self.cache).free >= needed+16*2**20:
                return
            if name in protected or path.suffix == '.parquet' or path.stem in self.maps:
                continue
            path.unlink(missing_ok=True); self.disk_bytes -= size; del self.disk[name]
        if shutil.disk_usage(self.cache).free < needed+16*2**20:
            raise RuntimeError('Insufficient Colab scratch space for the next vector window; existing Drive vectors retained.')

    def _block(self,ref):
        name = digest_json([ref['path'],ref['sha256']])
        if name not in self.maps:
            path = self.block_path(ref)
            self.io.fetch(self.data_root/ref['path'],path,ref['sha256'])
            values = np.load(path,allow_pickle=False,mmap_mode='r')
            self.io.validate(path,ref['sha256'],values,self.manifest['dimension'])
            self.maps[name] = values
            if len(self.maps)>4:
                self.maps.popitem(last=False)
        self.maps.move_to_end(name); self._touch(self.block_path(ref))
        return self.maps[name]

    def gather(self,positions):
        positions = np.asarray(positions,np.int64)
        if positions.ndim != 1 or np.any(positions<0) or np.any(positions>=self.manifest['input_count']):
            raise ValueError('Invalid content vector positions.')
        result = np.empty((len(positions),self.manifest['dimension']),np.float32)
        if not len(positions):
            return result
        selected = {}
        for out in np.argsort(positions,kind='stable'):
            index = int(np.searchsorted(self.starts,int(positions[out]),side='right'))-1
            selected.setdefault(index,[]).append((int(out),int(positions[out])-self.starts[index]))
        self.restore_mapping_bundle()
        indexes, groups = list(selected),{}
        def read_map(number,path):
            index = indexes[number]
            part = self.manifest['parts'][index]
            table = pq.read_table(path,columns=['path','sha256','row'])
            if table.num_rows != part['rows']:
                raise ValueError('Content map shape changed.')
            outputs,offsets = zip(*selected[index])
            for out,ref in zip(outputs,table.take(pa.array(offsets,type=pa.int64())).to_pylist(),strict=True):
                relative = PurePosixPath(ref['path'])
                if relative.is_absolute() or '..' in relative.parts or '\\' in ref['path'] or not relative.parts:
                    raise ValueError('Content reference escapes its declared corpus.')
                groups.setdefault((ref['path'],ref['sha256']),[]).append((out,ref['row']))
            self._touch(path)
        self.io.stage([(self.root/self.manifest['parts'][i]['path'],self.map_path(i),self.manifest['parts'][i]['sha256'])
            for i in indexes], 'Mapping cache',consume=read_map)
        self.save_mapping_bundle()
        references = list(groups)
        def read_block(number,path):
            relative, checksum = references[number]
            block = self._block({'path':relative,'sha256':checksum})
            outputs,offsets = zip(*groups[relative,checksum])
            if min(offsets)<0 or max(offsets)>=len(block):
                raise ValueError('Content map row escapes its vector block.')
            result[list(outputs)] = block[list(offsets)]
        self.io.stage([(self.data_root/name,self.block_path({'path':name,'sha256':sha}),sha)
            for name,sha in references], 'Vector cache',consume=read_block,reserve=self.reserve)
        return result

    def bundle_binding(self):
        return {'schema_version':1,'kind':'mapping-bundle-v1','data_folder_id':self.io.api.folder_id,
            'source_manifest_sha256':self.manifest['manifest_sha256']}

    def restore_mapping_bundle(self):
        if self._bundle_attempted or self.io.api is None or not hasattr(self.io.api,'publish_cache_checkpoint'):
            return
        self._bundle_attempted = True
        if all(self.map_path(i).is_file() for i in range(len(self.manifest['parts']))):
            return  # Preserve the old runtime's existing mapping cache.
        binding = self.bundle_binding()
        directory = 'retrieval/mapping_bundles/'+binding['source_manifest_sha256']
        saved = self.io.api.relative_json(directory+'/current.json')
        if saved is None:
            return
        checked(saved)
        if saved.get('binding') != binding or saved.get('parts') != len(self.manifest['parts']):
            raise ValueError('Mapping bundle differs from the current frozen view.')
        if saved.get('path') != 'maps-'+saved['sha256']+'.tar.gz':
            raise ValueError('Mapping bundle path changed.')
        payload = self.io.work/'mapping_bundles'/(saved['sha256']+'.tar.gz')
        self.io.fetch(self.data_root/directory/saved['path'],payload,saved['sha256'])
        expected = {p['path']:(i,p) for i,p in enumerate(self.manifest['parts'])}
        seen, total = set(),0
        with tarfile.open(payload,mode='r|gz') as archive:
            for member in archive:
                if not member.isfile() or member.name not in expected or member.name in seen or member.size>16*2**20:
                    raise ValueError('Unexpected file/link in mapping bundle.')
                total += member.size
                if total>512*2**20:
                    raise ValueError('Mapping bundle exceeds its bounded metadata budget.')
                i,p = expected[member.name]; target = self.map_path(i)
                if target.exists():
                    self.io.prove(target,p['sha256'])
                else:
                    target.parent.mkdir(parents=True,exist_ok=True)
                    stage = target.with_name('.'+uuid.uuid4().hex+'.part')
                    try:
                        src, sha = archive.extractfile(member),hashlib.sha256()
                        with src,stage.open('wb') as dst:
                            for chunk in iter(lambda:src.read(1024*1024),b''):
                                sha.update(chunk); dst.write(chunk)
                        if sha.hexdigest()!=p['sha256']:
                            raise ValueError('Mapping bundle member checksum changed.')
                        os.replace(stage,target); self.io.record(target,p['sha256'])
                    finally:
                        stage.unlink(missing_ok=True)
                seen.add(member.name)
        if seen != set(expected):
            raise ValueError('Mapping bundle has missing parts.')
        self._bundle_saved = True
        self.io.progress(f'Mapping bundle restored: {len(seen):,} parts from one Drive payload')

    def save_mapping_bundle(self):
        if (getattr(self,'_bundle_saved',False) or getattr(self,'_bundle_publish_attempted',False) or self.io.api is None
            or not hasattr(self.io.api,'publish_cache_checkpoint')
            or not all(self.io.known(self.map_path(i),p['sha256']) for i,p in enumerate(self.manifest['parts']))):
            return
        binding = self.bundle_binding()
        directory = 'retrieval/mapping_bundles/'+binding['source_manifest_sha256']
        local = self.io.work/'mapping_bundles'/(binding['source_manifest_sha256']+'.tar.gz')
        local.parent.mkdir(parents=True,exist_ok=True)
        if sum(self.map_path(i).stat().st_size for i in range(len(self.manifest['parts'])))>512*2**20:
            return  # Optional bundle never grows without a metadata bound.
        self._bundle_publish_attempted = True
        with tarfile.open(local,mode='w:gz',compresslevel=1) as archive:
            for i,p in enumerate(self.manifest['parts']):
                path = self.map_path(i)
                info = tarfile.TarInfo(p['path']); info.size = path.stat().st_size
                with path.open('rb') as stream:
                    archive.addfile(info,stream)
        checksum = sha256_file(local)
        saved = seal({'binding':binding,'path':'maps-'+checksum+'.tar.gz','sha256':checksum,
            'parts':len(self.manifest['parts'])})
        try:
            self.io.api.publish_cache_checkpoint(directory+'/current.json',local,saved)
            self._bundle_saved = True
            self.io.progress(f'Mapping bundle SAVED_ON_DRIVE: {saved["parts"]:,} parts; future runtimes restore one payload')
        except LookupPublishUnavailable:
            self.io.progress('Mapping bundle upload deferred; local mapping cache retained')


def index_sample(store,positions,io):
    """Checkpoint the exact deterministic ANN sample, not a second corpus copy."""
    positions = np.asarray(positions,np.int64)
    source = store.manifest['manifest_sha256']
    positions_sha = hashlib.sha256(positions.tobytes()).hexdigest()
    binding = {'schema_version':1,'kind':'index-sample-v1','source_manifest_sha256':source,
        'positions_sha256':positions_sha,'data_folder_id':getattr(io.api,'folder_id',None)}
    directory = 'retrieval/index_samples/'+source+'/'+positions_sha
    local = io.work/'index_samples'/digest_json([source,positions_sha])[:32]
    local.mkdir(parents=True,exist_ok=True)
    marker, payload = local/'sample.json',local/'sample.npy'
    saved = checked(read_json(marker)) if marker.is_file() else None
    if saved is None and io.api is not None and hasattr(io.api,'publish_cache_checkpoint'):
        saved = io.api.relative_json(directory+'/current.json')
        if saved is not None:
            checked(saved)
    if saved is not None:
        if saved.get('binding') != binding or saved.get('rows') != len(positions):
            raise ValueError('ANN sample checkpoint source/positions changed.')
        if saved.get('path') != 'sample-'+saved['sha256']+'.npy':
            raise ValueError('ANN sample checkpoint path changed.')
        io.fetch(store.data_root/directory/saved['path'],payload,saved['sha256'])
        values = np.load(payload,allow_pickle=False)
        io.validate(payload,saved['sha256'],values,store.manifest['dimension'])
        if len(values)!=len(positions):
            raise ValueError('ANN sample row count changed.')
        atomic_json(marker,saved)
        io.progress(f'ANN training sample reused: {len(values):,} vectors; no corpus read')
        return values
    io.progress(f'ANN sample preparation: {len(positions):,} existing vectors; bounded parallel payload reads')
    values = np.ascontiguousarray(store.gather(positions),dtype=np.float32)
    validate_vectors(values,len(positions),store.manifest['dimension'])
    stage = payload.with_suffix('.tmp.npy')
    np.save(stage,values,allow_pickle=False); os.replace(stage,payload)
    checksum = sha256_file(payload); io.record(payload,checksum,shape=values.shape)
    saved = seal({'binding':binding,'path':'sample-'+checksum+'.npy','sha256':checksum,
        'parts':1,'rows':len(positions),'dimension':store.manifest['dimension']})
    atomic_json(marker,saved)
    if io.api is not None and hasattr(io.api,'publish_cache_checkpoint'):
        try:
            io.api.publish_cache_checkpoint(directory+'/current.json',payload,saved)
            io.progress(f'ANN sample SAVED_ON_DRIVE: {len(values):,} vectors; source vectors retained')
        except LookupPublishUnavailable:
            io.progress('ANN sample upload deferred; local sample retained')
    return values

"""Bounded parallel map checks and visible GPU startup; producer files stay pinned."""
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import gzip
from pathlib import Path, PurePosixPath
import shutil
import sqlite3
import sys
import threading
import time

from vietmedbridge.artifacts import (atomic_json, digest_json, local_workspace, publish_file,
    read_json, sha256_file, utc_now, verify_file)
from vietmedbridge.content_embeddings import ContentBlocks as NativeContentBlocks, checked, seal
from vietmedbridge.portable_embeddings import resources_relative
from vietmedbridge.scale_vectors import bound

# Notebook cells embed the pure-stdlib mount helper before this adapter. Module
# imports in local tests load the same helper from scripts, outside producer src.
if 'safe_colab_remount' not in globals():
    from scripts.colab_drive_mount import safe_colab_remount
if 'authenticate_drive_receipts' not in globals():
    from scripts.drive_receipts_api import authenticate_drive_receipts, LookupPublishUnavailable
if 'SubmissionVectorParts' not in globals():
    from scripts.submission_vector_io import (SubmissionVectorParts, VerifiedSubmissionIO,
        read_declared_vector_manifest, index_sample)


class ResumableContentBlocks(NativeContentBlocks):
    """Retain validated receipt lookups on local disk across Drive recovery.

    The native content key, encoder identity and vector payload validation stay
    unchanged. A compressed lookup is an optimization, never a coverage receipt.
    """
    def __init__(self, data_root, encoder, work_dir, *, progress=print, receipt_api=None, inventory=True):
        self.receipt_api = receipt_api
        if receipt_api is not None and Path(data_root).absolute() not in (receipt_api.requested_root, receipt_api.data_root):
            raise ValueError('Receipt API root differs from the embedding data root.')
        self.root = receipt_api.data_root if receipt_api is not None else Path(data_root).resolve()
        self.encoder = encoder
        self.identity = {'encoder': encoder.identity, 'dimension': encoder.dimension,
            'format': 'content-float32-l2-v1'}
        self.directory = self.root / 'model_cache' / digest_json(self.identity)[:24]
        if receipt_api is None:
            self.directory.mkdir(parents=True, exist_ok=True)
        self.progress = progress
        self.binding = {'schema_version': 1, 'data_root': str(self.root), 'identity': self.identity}
        self.lookup = Path(work_dir) / 'receipt_lookups' / digest_json(self.binding)[:16]
        self.lookup.mkdir(parents=True, exist_ok=True)
        self.database = self.lookup / 'lookup.sqlite'
        self.snapshot_binding = (dict(schema_version=2, data_folder_id=receipt_api.folder_id, identity=self.identity)
            if receipt_api is not None and hasattr(receipt_api, 'folder_id') else self.binding)
        self.remote = self.root / 'retrieval/content_lookups' / digest_json(self.snapshot_binding)[:16]
        self.legacy_remote = self.root / 'retrieval/content_lookups' / digest_json(self.binding)[:16]
        self.temporary = local_workspace(work_dir)
        self.local = Path(self.temporary.__enter__())
        self.con = None
        self._snapshot_dirty = False
        self._snapshot_busy = False
        self._next_snapshot_count = 256
        self._receipt_count = 0
        try:
            if not self.database.exists():
                if not inventory:
                    raise FileNotFoundError('No local lookup to save; no receipt scan or model inference was started.')
                self._restore()
            self.con = sqlite3.connect(self.database)
            self.con.execute('PRAGMA cache_size=-32768')
            self.con.execute('PRAGMA synchronous=FULL')
            self.con.execute('CREATE TABLE IF NOT EXISTS metadata(value TEXT NOT NULL)')
            row = self.con.execute('SELECT value FROM metadata').fetchone()
            signature = digest_json(self.binding)
            if row is not None and row[0] != signature:
                raise ValueError('Receipt lookup model/root binding changed.')
            if row is None:
                self.con.execute('INSERT INTO metadata VALUES (?)', [signature])
            self.con.execute('CREATE TABLE IF NOT EXISTS vectors(key TEXT PRIMARY KEY,path TEXT,sha TEXT,row INTEGER)')
            self.con.execute('CREATE TABLE IF NOT EXISTS receipts(name TEXT PRIMARY KEY,digest TEXT NOT NULL)')
            self.con.commit()
            known = {row[0] for row in self.con.execute('SELECT name FROM receipts')}
            self._receipt_count = len(known)
            self._next_snapshot_count = max(256, len(known)*2)
            if not inventory:
                self._snapshot_dirty = True
                self._snapshot()
                return
            self.progress(f'BGE receipt inventory: retained {len(known):,} validated receipts; listing new names')
            markers = (receipt_api.receipts(self.identity) if receipt_api is not None
                else sorted(self.directory.glob('*.done.json')))
            pending = [path for path in markers if path.name not in known]
            self.progress(f'BGE receipt inventory: {len(markers)-len(pending):,}/{len(markers):,} reused'
                f' | {len(pending):,} unread | {"API" if receipt_api is not None else "mount"} transport | commit each receipt')
            self._read_pending(pending, len(markers)-len(pending), len(markers))
            self.progress(f'BGE receipt inventory ready: {len(markers):,} receipts | native vector SHA checked on use')
            if receipt_api is not None:
                if pending:
                    self._snapshot_dirty = True
                if hasattr(receipt_api, 'publish_lookup') and not self._snapshot_dirty:
                    saved = receipt_api.relative_json((self.remote / 'current.json').relative_to(self.root).as_posix())
                    if saved is None:
                        self._snapshot_dirty = True
                    else:
                        checked(saved)
                        if saved.get('binding') != self.snapshot_binding:
                            raise ValueError('Published lookup producer/folder changed.')
                        self._snapshot_dirty = saved['receipts'] < self._receipt_count
                if self._snapshot_dirty:
                    self._snapshot()
                self.progress('BGE API receipt lookup ready; no mount I/O for inventory')
            elif pending or not (self.remote / 'current.json').is_file():
                self._snapshot()
        except BaseException:
            self.close()
            raise

    def _read_pending(self, pending, reused, total):
        # Keep at most two reads in flight: pool.map queued the whole directory
        # and discarded its temporary SQLite lookup after any mount failure.
        readers = 4 if self.receipt_api is not None else 2
        pool = ThreadPoolExecutor(max_workers=readers)
        iterator, futures = iter(pending), []
        count, last_log = 0, time.monotonic()
        def refill():
            while len(futures) < readers:
                path = next(iterator, None)
                if path is None:
                    break
                futures.append((path, pool.submit(lambda p: checked(
                    p.read_json() if self.receipt_api is not None else read_json(p)), path)))
        try:
            refill()
            while futures:
                # Register in native filename order, preserving the first
                # existing vector reference when blocks contain overlapping text.
                path, future = futures.pop(0)
                block = future.result()
                if path.name != 'block-' + digest_json(block['keys']) + '.done.json':
                    raise ValueError('Content receipt name/key binding changed.')
                self.register(block)
                count += 1
                if count % 32 == 0 or time.monotonic()-last_log >= 15 or count == len(pending):
                    self.progress(f'BGE receipt inventory: {reused+count:,}/{total:,}'
                        f' | reused {reused:,}, newly indexed {count:,} | local checkpoint committed')
                    last_log = time.monotonic()
                refill()
        except BaseException:
            pool.shutdown(wait=False, cancel_futures=True)
            raise
        else:
            pool.shutdown(wait=True)

    def register(self, block):
        checked(block)
        marker = 'block-' + digest_json(block['keys']) + '.done.json'
        old = self.con.execute('SELECT digest FROM receipts WHERE name=?', [marker]).fetchone()
        if old is not None and old[0] != block['manifest_sha256']:
            raise ValueError('Immutable content receipt changed in the retained lookup.')
        try:
            # Keys and their receipt commit together; interrupted reads never
            # advance the inventory or erase an earlier committed prefix.
            if self.receipt_api is None:
                super().register(block)
            else:
                if block['identity'] != self.identity or len(block['keys']) != block['rows']:
                    raise ValueError('Content vector producer/key mismatch.')
                relative = PurePosixPath(block['path'])
                if relative.is_absolute() or '..' in relative.parts or '\\' in block['path'] or not relative.parts:
                    raise ValueError('API content reference escapes its data root.')
                # No filesystem resolution through FUSE during API inventory.
                # Native vector readers still resolve the actual path and check
                # its SHA256/shape before consuming the vector payload.
                self.con.executemany('INSERT OR IGNORE INTO vectors VALUES (?,?,?,?)',
                    [(key,block['path'],block['sha256'],i) for i,key in enumerate(block['keys'])])
            self.con.execute('INSERT OR IGNORE INTO receipts VALUES (?,?)', [marker, block['manifest_sha256']])
            self.con.commit()
            if old is None:
                self._receipt_count += 1
                self._snapshot_dirty = True
        except BaseException:
            self.con.rollback()
            raise
        if (self.receipt_api is not None and self._receipt_count >= self._next_snapshot_count
            and hasattr(self.receipt_api, 'publish_lookup')):
            self._next_snapshot_count = max(256, self._receipt_count*2)
            self._snapshot()

    def _restore(self):
        remote = self.remote
        marker = remote / 'current.json'
        if self.receipt_api is None:
            if not marker.is_file():
                return
            saved = checked(read_json(marker))
        else:
            value = self.receipt_api.relative_json(marker.relative_to(self.root).as_posix())
            if value is None and self.legacy_remote != self.remote:
                remote = self.legacy_remote
                value = self.receipt_api.relative_json((remote / 'current.json').relative_to(self.root).as_posix())
            if value is None:
                return
            saved = checked(value)
        if saved.get('binding') != (self.snapshot_binding if remote == self.remote else self.binding):
            raise ValueError('Published receipt lookup producer/root changed.')
        relative = PurePosixPath(saved['path'])
        if relative.is_absolute() or len(relative.parts) != 1 or '\\' in saved['path']:
            raise ValueError('Published receipt snapshot path escapes its directory.')
        self.progress('BGE receipt inventory: restore one compressed lookup; no old receipt JSON scan')
        local = self.local / 'lookup.sqlite.gz'
        if self.receipt_api is None:
            shutil.copyfile(bound(remote, saved['path']), local)
        else:
            source = self.receipt_api.relative((remote / saved['path']).relative_to(self.root).as_posix())
            if source is None:
                raise ValueError('Published API receipt lookup payload is missing.')
            self.receipt_api.download(source, local, expected_sha256=saved['sha256'])
        verify_file(local, saved['sha256'])
        stage = self.lookup / 'restore.sqlite'
        try:
            with gzip.open(local, 'rb') as src, stage.open('wb') as dst:
                shutil.copyfileobj(src, dst, length=1024*1024)
            verify_file(stage, saved['sqlite_sha256'])
            # A shared folder has the same identity on another account even when
            # Colab exposes it through a different MyDrive/shortcut path.
            old_binding = saved.get('lookup_binding', saved['binding'])
            restored = sqlite3.connect(stage)
            try:
                rows = restored.execute('SELECT value FROM metadata').fetchall()
                if rows != [(digest_json(old_binding),)] or old_binding.get('identity') != self.identity:
                    raise ValueError('Restored lookup SQLite binding changed.')
                if restored.execute('SELECT count(*) FROM receipts').fetchone()[0] != saved['receipts']:
                    raise ValueError('Restored lookup receipt count changed.')
                if old_binding != self.binding:
                    restored.execute('UPDATE metadata SET value=?', [digest_json(self.binding)])
                    restored.commit()
            finally:
                restored.close()
            stage.replace(self.database)
        finally:
            stage.unlink(missing_ok=True)

    def _snapshot(self):
        if self._snapshot_busy:
            return
        if self.receipt_api is not None and not hasattr(self.receipt_api, 'publish_lookup'):
            return  # Compatibility with older read-only transports; local prefix remains usable.
        self._snapshot_busy = True
        self.progress('BGE receipt inventory: save compressed lookup for future runtimes')
        copy = self.local / 'lookup-backup.sqlite'
        payload = self.local / 'lookup.sqlite.gz'
        try:
            backup = sqlite3.connect(copy)
            try:
                self.con.backup(backup)
            finally:
                backup.close()
            checksum = sha256_file(copy)
            with copy.open('rb') as src, gzip.open(payload, 'wb', compresslevel=1) as dst:
                shutil.copyfileobj(src, dst, length=1024*1024)
            target = self.remote / ('lookup-' + sha256_file(payload) + '.sqlite.gz')
            record = seal({'binding': self.snapshot_binding, 'lookup_binding': self.binding, 'path': target.name,
                'sha256': sha256_file(payload), 'sqlite_sha256': checksum,
                'receipts': self.con.execute('SELECT count(*) FROM receipts').fetchone()[0],
                'updated_at': utc_now(),
                'scope': 'Receipt lookup only; vector payloads remain subject to native SHA/shape checks'})
            if self.receipt_api is None:
                publish_file(payload, target)
                atomic_json(self.remote / 'current.json', record)
            else:
                self.receipt_api.publish_lookup((self.remote / 'current.json').relative_to(self.root).as_posix(),
                    payload, record)
            self._snapshot_dirty = False
            self.progress(f"BGE receipt lookup SAVED_ON_DRIVE: {record['receipts']:,} receipts; next runtime restores this lookup")
        except (OSError, LookupPublishUnavailable) as error:
            # Optional acceleration only: the local committed database survives,
            # and native reads/writes still surface actual mount/space failures.
            self.progress(f'BGE receipt lookup snapshot deferred: {type(error).__name__}; local checkpoint retained')
        finally:
            self._snapshot_busy = False
            copy.unlink(missing_ok=True)
            payload.unlink(missing_ok=True)

    def close(self):
        if self.con is not None:
            try:
                if self.receipt_api is not None and self._snapshot_dirty:
                    self._snapshot()
            finally:
                self.con.close()
                self.con = None
        self.temporary.__exit__(None, None, None)


def read_bge_maps_parallel(data_root, candidate, inputs, config, *, progress=print):
    """Same BGE identity/map/payload checks as imported_content_manifest, in frozen order."""
    view = Path(data_root) / resources_relative(candidate, config) / 'bge'
    path = view / 'config.json'
    if not path.is_file():
        return view, None
    identity = read_json(path)
    if (identity.get('input_manifest_sha256') != inputs['manifest_sha256']
        or identity.get('input_count') != inputs['input_count']
        or identity.get('format') != 'content-reference-v1'
        or identity.get('dimension') != 1024
        or any(identity.get('encoder', {}).get(k) != v for k, v in config['dense'].items())):
        raise ValueError('Imported BGE view differs from the active frozen corpus/model.')
    signature = digest_json(identity)
    sources = inputs['parts']
    progress(f'BGE mapping check: 0/{len(sources):,} | up to 8 readers | SHA256 preserved')

    def check(source):
        marker = view / f"map-{source['start']:012d}.done.json"
        if not marker.is_file():
            return None
        saved = checked(read_json(marker))
        expected = {'signature': signature, 'start': source['start'], 'rows': source['rows'],
            'input_sha256': source['sha256'], 'path': f"map-{source['start']:012d}.parquet"}
        if any(saved.get(k) != v for k, v in expected.items()):
            raise ValueError('Imported BGE mapping differs from the frozen input order.')
        verify_file(view / saved['path'], saved['sha256'])
        return saved

    ordered = [None] * len(sources)
    found = 0
    pool = ThreadPoolExecutor(max_workers=min(8, max(1, len(sources))))
    try:
        futures = {pool.submit(check, source): i for i, source in enumerate(sources)}
        for count, future in enumerate(as_completed(futures), 1):
            saved = future.result()
            ordered[futures[future]] = saved
            found += saved is not None
            if count % 32 == 0 or count == len(sources):
                progress(f'BGE mapping check: {count:,}/{len(sources):,} | valid maps {found:,}')
    except BaseException:
        # Do not trap the notebook in executor shutdown if Drive readers stall.
        # In-flight work only reads files; it never publishes or changes checkpoints.
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    else:
        pool.shutdown(wait=True)
    parts = [p for p in ordered if p is not None]
    return view, seal(identity | {'signature': signature, 'parts': parts,
        'state': 'COMPLETE' if len(parts) == len(sources) else 'IN_PROGRESS'})


def ensure_submission_mount(data_root, expected_candidate_sha256):
    """Return the matching root through known Drive aliases; JSON only."""
    from vietmedbridge.colab_storage import _disconnected, _drive_mountpoint
    from vietmedbridge.scale_benchmark import load_handoff
    if not isinstance(expected_candidate_sha256, str) or not expected_candidate_sha256:
        raise ValueError('Expected frozen candidate is required before reconnecting.')
    print('SUBMISSION preflight: read frozen handoff JSON; preparation is not repeated', flush=True)
    root = Path(data_root)
    mount = _drive_mountpoint(root)
    # These are observed aliases of this shared project, not a Drive-wide search.
    aliases = [root]
    if mount is not None:
        aliases += [Path('/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge/data'),
            Path('/content/drive/MyDrive/VietMedBridge/data')]
    aliases = list(dict.fromkeys(aliases))
    missing = None
    for attempt in range(2 if mount is not None else 1):
        for location in aliases:
            try:
                _, candidate, _ = load_handoff(location)
            except FileNotFoundError as error:
                missing = error
                continue
            except OSError as error:
                if mount is None or not _disconnected(error, mount):
                    raise
                if attempt:
                    raise
                break
            else:
                if candidate['candidate_manifest_sha256'] != expected_candidate_sha256:
                    raise ValueError('Active candidate changed; refusing to resume another submission corpus.')
                print('SUBMISSION preflight: same frozen candidate verified | DATA_ROOT:', location, flush=True)
                return location
        if mount is not None and not attempt:
            print('SUBMISSION preflight: handoff unavailable; reconnect Drive, retain runtime and local lookup', flush=True)
            safe_colab_remount(mount, readonly=False)
    if mount is None and missing is not None:
        raise missing
    raise FileNotFoundError('Frozen submission handoff not found after reconnect. Checked only: '
        + ', '.join(map(str, aliases)) + '. No data or checkpoint was created.')


def prepare_drive_receipt_lookup(receipt_api, checkout, work_dir, *, progress=print):
    """CPU metadata only, before mapping scans or any neural model allocation."""
    from types import SimpleNamespace
    config = read_json(Path(checkout) / 'configs/retrieval_full.json')
    relative = Path(resources_relative(receipt_api.candidate, config)) / 'bge/config.json'
    header = receipt_api.relative_json(relative.as_posix())
    if header is None:
        progress('BGE API lookup: no existing BGE view; no paid receipt inventory to prepare')
        return
    if (header.get('input_manifest_sha256') != receipt_api.active['index_inputs_manifest_sha256']
        or header.get('dimension') != 1024 or header.get('format') != 'content-reference-v1'
        or any(header.get('encoder', {}).get(k) != v for k,v in config['dense'].items())):
        raise ValueError('API BGE producer/header differs from the active frozen inputs.')
    # A completed, sealed view already carries every map's source/checksum.
    # Do not construct the content-key receipt inventory just to read that view.
    build_run = receipt_api.active.get('build_run')
    inputs = receipt_api.relative_json('processed/'+build_run+'/index_inputs/units.json') if build_run else None
    if inputs is not None:
        if inputs.get('manifest_sha256') != receipt_api.active['index_inputs_manifest_sha256']:
            raise ValueError('Frozen input manifest differs from the active handoff.')
        declared = read_declared_vector_manifest(receipt_api.requested_root,receipt_api.candidate,
            inputs,config,api=receipt_api,progress=progress)
        if declared is not None:
            progress('BGE COMPLETE: skip content receipt inventory and per-map preflight; validate payload on use')
            return
    progress('BGE API lookup preparation: CPU metadata only; no mapping payload scan or model weights')
    encoder = SimpleNamespace(identity=header['encoder'], dimension=header['dimension'])
    lookup = ResumableContentBlocks(receipt_api.requested_root, encoder, work_dir,
        progress=progress, receipt_api=receipt_api)
    lookup.close()
    progress('BGE API receipt lookup ready; continuing native submission checks')


def save_existing_drive_receipt_lookup(receipt_api, checkout, work_dir, *, progress=print):
    """Save an older cell's committed local lookup, without listing receipts or encoding."""
    from types import SimpleNamespace
    config = read_json(Path(checkout) / 'configs/retrieval_full.json')
    relative = Path(resources_relative(receipt_api.candidate, config)) / 'bge/config.json'
    header = receipt_api.relative_json(relative.as_posix())
    if (header is None or header.get('input_manifest_sha256') != receipt_api.active['index_inputs_manifest_sha256']
        or header.get('dimension') != 1024 or header.get('format') != 'content-reference-v1'
        or any(header.get('encoder', {}).get(k) != v for k,v in config['dense'].items())):
        raise ValueError('Saved lookup producer differs from the current frozen BGE view.')
    encoder = SimpleNamespace(identity=header['encoder'], dimension=header['dimension'])
    lookup = ResumableContentBlocks(receipt_api.requested_root, encoder, work_dir,
        progress=progress, receipt_api=receipt_api, inventory=False)
    lookup.close()
    if lookup._snapshot_dirty:
        raise LookupPublishUnavailable('Lookup remains local; Drive save was not confirmed. Keep this runtime.')


class SubmissionProgress:
    def __init__(self, data_root, work_dir, label, *, interval=15, drive_reports=True):
        self.root, self.work, self.label = Path(data_root), Path(work_dir), label
        self.interval, self.main_id = interval, threading.get_ident()
        self.started, self.last_saved = time.monotonic(), 0
        self.phase = 'Entering submission workflow'
        self.binding = {}
        self.drive_reports = drive_reports
        self.stop = threading.Event()

    def location(self):
        frame = sys._current_frames().get(self.main_id)
        location = None
        while frame is not None:
            if frame.f_code.co_name in {'read_json', 'sha256_file'}:
                value = frame.f_locals.get('path')
                if isinstance(value, (str, Path)):
                    location = str(value)
                    break
            if location is None and 'vietmedbridge' in frame.f_code.co_filename:
                location = Path(frame.f_code.co_filename).name + ':' + str(frame.f_lineno)
            frame = frame.f_back
        return location

    def snapshot(self, state='RUNNING'):
        return {'state': state, 'variant': self.label, 'phase': self.phase,
            'elapsed_seconds': round(time.monotonic() - self.started, 1),
            'waiting_at': self.location(), 'updated_at': utc_now(),
            **self.binding,
            'scope': 'Live stage diagnostic; not an embedding completion or submission receipt'}

    def report(self, message):
        self.phase = message
        print(message, flush=True)
        if self.drive_reports and time.monotonic() - self.last_saved >= 15:
            try:
                atomic_json(self.root / 'retrieval/submission_progress.json', self.snapshot())
                self.last_saved = time.monotonic()
            except OSError:
                pass

    def heartbeat(self):
        while not self.stop.wait(self.interval):
            status = self.snapshot()
            print(f"SUBMISSION heartbeat | {status['elapsed_seconds']:.0f}s | {self.phase}"
                f" | waiting at {status['waiting_at']}", flush=True)
            try:
                atomic_json(self.work / 'submission_progress.json', status)
            except OSError:
                pass  # Diagnostics cannot replace or invalidate native checkpoint errors.


@contextmanager
def submission_gpu_progress(data_root, work_dir, *, label='baseline', receipt_api=None):
    """Single-writer notebook adapter; restores every patched binding on exit."""
    from vietmedbridge import (colab_storage, content_embeddings, scale_baseline, submission_workflow,
        full_scale_runtime, cached_strong_runtime, vector_search)
    progress = SubmissionProgress(data_root, work_dir, label, drive_reports=receipt_api is None)
    original_maps = scale_baseline.imported_bge_manifest
    original_blocks = content_embeddings.ContentBlocks
    original_remount = colab_storage._remount
    io = VerifiedSubmissionIO(work_dir,api=receipt_api,progress=progress.report)
    vector_modules = (content_embeddings,scale_baseline,full_scale_runtime,cached_strong_runtime)
    vector_originals = [(module,module.ContentVectorParts) for module in vector_modules]
    content_originals = [(module,module.imported_content_manifest)
        for module in (scale_baseline,cached_strong_runtime)]
    original_content = scale_baseline.imported_content_manifest
    original_build = vector_search.build_faiss_index
    search_originals = [(module,module.search_vectors)
        for module in (vector_search,full_scale_runtime,cached_strong_runtime)]
    original_search = vector_search.search_vectors
    names = ('_submission_context', '_saved_submission', '_release_catalog_staging', '_ensure_primary')
    originals = {name: getattr(submission_workflow, name) for name in names}

    def wrap(name, function):
        def observed(*args, **kwargs):
            progress.report('SUBMISSION stage: ' + name)
            result = function(*args, **kwargs)
            if name == '_submission_context':
                progress.binding = {'build_run': result[3].name,
                    'candidate_manifest_sha256': result[4]['candidate_manifest_sha256'],
                    'input_count': result[5]['input_count']}
            progress.report('SUBMISSION stage complete: ' + name)
            return result
        return observed

    def content_manifest(*args, family):
        result = read_declared_vector_manifest(*args,family=family,api=receipt_api,progress=progress.report)
        if result is not None:
            return result
        if family == 'bge':
            return read_bge_maps_parallel(*args,progress=progress.report)
        return original_content(*args,family=family)

    class SampleStore:
        def __init__(self,store):
            self.store = store
        def __getattr__(self,name):
            return getattr(self.store,name)
        def gather(self,positions):
            return index_sample(self.store,positions,io)

    def build_index(store,*args,**kwargs):
        progress.report('FAISS index: reuse completed shards; checkpoint exact training sample')
        if isinstance(store,SubmissionVectorParts):
            store = SampleStore(store)
        return original_build(store,*args,**kwargs)

    def search(*args,**kwargs):
        progress.report('Vector search: cached corpus vectors; native ANN recall check and query checkpoints')
        return original_search(*args,**kwargs)

    for module,_ in vector_originals:
        module.ContentVectorParts = lambda *a, **k: SubmissionVectorParts(*a,io=io,**k)
    for module,_ in content_originals:
        module.imported_content_manifest = content_manifest
    for module,_ in search_originals:
        module.search_vectors = search
    vector_search.build_faiss_index = build_index
    scale_baseline.imported_bge_manifest = lambda *a, **k: content_manifest(*a,**k,family='bge')
    def resumable_content_blocks(*args, **kwargs):
        data_root = args[0] if args else kwargs.get('data_root')
        api = receipt_api
        if api is not None:
            root = Path(data_root)
            allowed = (Path(api.requested_root), Path(api.data_root))
            if all(root.absolute() != item and root.resolve() != item.resolve() for item in allowed):
                api = None  # Generated document vectors live under output_root, not the source corpus.
        kwargs['progress'], kwargs['receipt_api'] = progress.report, api
        return ResumableContentBlocks(*args, **kwargs)
    content_embeddings.ContentBlocks = resumable_content_blocks
    colab_storage._remount = safe_colab_remount
    for name, function in originals.items():
        setattr(submission_workflow, name, wrap(name, function))
    thread = threading.Thread(target=progress.heartbeat, name='submission-progress', daemon=True)
    progress.report('SUBMISSION startup | live progress every 15 seconds | same candidate and checkpoints')
    thread.start()
    state = 'STOPPED'
    try:
        yield progress
        state = 'CALL_FINISHED'
    finally:
        progress.stop.set()
        thread.join(timeout=1)
        scale_baseline.imported_bge_manifest = original_maps
        content_embeddings.ContentBlocks = original_blocks
        colab_storage._remount = original_remount
        for module,function in vector_originals:
            module.ContentVectorParts = function
        for module,function in content_originals:
            module.imported_content_manifest = function
        for module,function in search_originals:
            module.search_vectors = function
        vector_search.build_faiss_index = original_build
        for name, function in originals.items():
            setattr(submission_workflow, name, function)
        status = progress.snapshot(state)
        try:
            atomic_json(Path(work_dir) / 'submission_progress.json', status)
        except OSError:
            pass
        if receipt_api is None or state == 'CALL_FINISHED':
            try:
                atomic_json(Path(data_root) / 'retrieval/submission_progress.json', status)
            except OSError:
                pass


from vietmedbridge.submission_workflow import run_current_submission
STRONG = None
if GPU_SESSION:
    DATA_ROOT = ensure_submission_mount(DATA_ROOT, PREPARATION["candidate_manifest_sha256"])
    DRIVE_RECEIPTS = authenticate_drive_receipts(DATA_ROOT, PREPARATION["candidate_manifest_sha256"])
    with submission_gpu_progress(DATA_ROOT, WORK_DIR, label="strong", receipt_api=DRIVE_RECEIPTS) as receipt_progress:
        prepare_drive_receipt_lookup(DRIVE_RECEIPTS, CHECKOUT, WORK_DIR, progress=receipt_progress.report)
        STRONG = run_current_submission(DATA_ROOT,CHECKOUT,WORK_DIR,code_commit=COMMIT)
    print(json.dumps(STRONG["ready"],ensure_ascii=False,indent=2),flush=True)
    print("Diagnostics:",STRONG["diagnostics_path"],flush=True)
    if STRONG["ready"]["state"] != "READY_FOR_MANUAL_UPLOAD":
        print("Bản strong chưa hoàn tất; baseline ZIP vẫn còn trên Drive. Run all cùng notebook để tiếp tục checkpoint.",flush=True)
else:
    print("GPU phase skipped on CPU.")


In [ ]:
#@title 5. Xem vị trí ZIP đã kiểm tra trên Drive (không tải về máy)
latest = DATA_ROOT / "retrieval/latest_submission.json"
if latest.is_file():
    submission = read_json(latest)
    if submission["candidate_manifest_sha256"] != PREPARATION["candidate_manifest_sha256"]:
        raise ValueError("ZIP thuộc candidate khác; chưa tải nhầm kết quả cũ.")
    verify_file(submission["zip_path"],submission["zip_sha256"])
    print(json.dumps(submission,ensure_ascii=False,indent=2),flush=True)
    print("ZIP lưu trên Drive:", submission["zip_path"], flush=True)
    print("Notebook chỉ hiển thị vị trí file; không tự tải về máy tính.",flush=True)
else:
    print("Chưa có ZIP hoàn tất. CPU preparation không phải submission; tiếp tục pha GPU.",flush=True)
